[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/pub-dialogue/blob/main/public_dialogue_analyser_v12b_4.ipynb)

# Public Dialogue Analyser

This notebook analyses **66 UK public dialogue reports (2004–2025)** covering AI, gene editing,
nanotechnology, nuclear energy, geoengineering, drones, and quantum technologies.

**Research question:** What concerns and benefits do members of the public consistently
raise across different technologies — and how cross-cutting are those themes?

**Pipeline overview:**
1. Load 66 PDF reports and chunk them into paragraphs
2. Use an LLM (GPT-4o) to extract *concern* and *benefit* phrases from each paragraph
3. Embed phrases semantically and cluster into themes (k-means)
4. Label clusters with the LLM and classify as cross-cutting vs. technology-specific
5. Characterise the AI dialogue footprint, temporal trends, and risk–benefit balance

**Parts:**
- **Part I** — Concern extraction, clustering, and AI analysis
- **Part II** — Benefit extraction, clustering, and AI analysis  
- **Part III** — Risk–benefit comparison
- **Part IV** — Robustness and sensitivity checks
- **Part V** — Export all outputs

# Part I: Concern analysis

This part identifies and characterises the *concerns* participants raised across the 66 public
dialogue reports. The pipeline runs in five stages: setup → corpus loading →
LLM extraction → embedding + clustering → labelling and analysis.

## Stage 0 — Setup and configuration

The setup cells install Python dependencies, import shared utilities from `dialogue_utils.py`,
and configure the OpenAI API key. When running in Google Colab for the first time, `dialogue_utils.py`
is automatically downloaded from the repository.

**Key configuration variables** (set in the "Import libraries" cell):
- `OUTPUT_FOLDER` — where CSV/JSON outputs are written
- `LLM_MODEL` — OpenAI model used for extraction and labelling (default `gpt-4o`)
- `N_CONCERN_CLUSTERS` — number of k-means clusters for concern phrases

In [ ]:
# @title Install pub-dialogue package
# Local: run 'pip install -e .' once in the repo root.
# Colab:  installs directly from GitHub.
try:
    import pub_dialogue
except ImportError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/mlatcl/pub-dialogue.git",
    ])
    import pub_dialogue

In [ ]:
# @title Import libraries and define configuration
import os
import json
import time
import re
import random
from pathlib import Path
from collections import Counter
from datetime import datetime
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import fitz  # PyMuPDF
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from openai import OpenAI
from IPython.display import display, HTML

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Model Configuration
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-5-mini"

# Clustering - now 75 clusters for concern phrases (smaller units than paragraphs)
N_CONCERN_CLUSTERS = 75
N_BENEFIT_CLUSTERS = 75
SOFT_MEMBERSHIP_THRESHOLD = 0.3

# Chunking (same as v8)
MIN_CHUNK_WORDS = 50
MAX_CHUNK_WORDS = 500
MIN_CHUNK_CHARS = 100

# Processing
EMBEDDING_BATCH_SIZE = 100
EXTRACTION_BATCH_SIZE = 10  # For concern extraction

# Paths
PDF_FOLDER = Path("/content/pdfs")
OUTPUT_FOLDER = Path("/content/outputs")
CHECKPOINT_FOLDER = Path("/content/checkpoints")

for folder in [PDF_FOLDER, OUTPUT_FOLDER, CHECKPOINT_FOLDER]:
    folder.mkdir(exist_ok=True)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150

import warnings
warnings.filterwarnings('ignore')

print("Configuration loaded")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Target clusters: {N_CONCERN_CLUSTERS}")
print(f"  LLM for extraction: {LLM_MODEL}")

from dialogue_utils import *  # noqa: F401,F403

In [ ]:
# show_status, show_complete, show_warning, save_checkpoint, load_checkpoint
# — imported from dialogue_utils


In [ ]:
# @title ☁️  Google Drive checkpoint management
# ---------------------------------------------------------------------------
# The full analysis takes ~5 hours (LLM extraction + embeddings).
# Run this cell to save all checkpoints to Google Drive so they survive
# a Colab runtime disconnect.  Skip it to use session-local storage only.
# ---------------------------------------------------------------------------

USE_DRIVE_CHECKPOINTS = False   # @param {type:"boolean"}
DRIVE_FOLDER_NAME = "pub-dialogue-checkpoints"  # @param {type:"string"}

if USE_DRIVE_CHECKPOINTS:
    try:
        from google.colab import drive as _drive
        _drive.mount('/content/drive', force_remount=False)
        _drive_cp = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}")
        _drive_cp.mkdir(parents=True, exist_ok=True)
        # Override CHECKPOINT_FOLDER defined in the config cell
        CHECKPOINT_FOLDER = _drive_cp
        show_complete(f"Checkpoints -> Google Drive: {CHECKPOINT_FOLDER}")
    except Exception as _e:
        show_warning(f"Drive mount failed ({_e}). Using session-local checkpoints.")
        show_warning("Checkpoints WILL BE LOST if this runtime disconnects.")
else:
    show_warning(f"Session-local checkpoints: {CHECKPOINT_FOLDER}")
    show_warning("These will be lost on runtime disconnect.")

# ── Checkpoint status dashboard ─────────────────────────────────────────────
print()
print("-" * 60)
print("  Checkpoint status")
print("-" * 60)

_stages = [
    ("PDF chunks (chunks_df)",          CHECKPOINT_FOLDER / "chunks_df.parquet"),
    ("Concern extraction (~4 h)",        CHECKPOINT_FOLDER / "extracted_concerns.json"),
    ("Concern embeddings (~30 m)",       CHECKPOINT_FOLDER / "concern_embeddings.npy"),
    ("Benefit extraction (~4 h)",        CHECKPOINT_FOLDER / "extracted_benefits.json"),
    ("Benefit embeddings (~30 m)",       CHECKPOINT_FOLDER / "benefit_embeddings.npy"),
    ("Concern cluster labels",           OUTPUT_FOLDER / f"concern_labels_k{N_CONCERN_CLUSTERS}.json"),
    ("Benefit cluster labels",           OUTPUT_FOLDER / f"benefit_labels_k{N_BENEFIT_CLUSTERS}.json"),
]

for _name, _path in _stages:
    _status = "cached   " if _path.exists() else "not yet computed"
    _tick   = "OK" if _path.exists() else "--"
    print(f"  [{_tick}]  {_name:<38s}  {_status}")

print("-" * 60)
print("Cells with cached checkpoints will be skipped automatically.")

In [ ]:
# @title Configure API access
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    print("API key loaded from Colab secrets")
except:
    from getpass import getpass
    api_key = getpass("Enter OpenAI API key: ")
    print("API key entered manually")

client = OpenAI(api_key=api_key)

try:
    client.models.list()
    print("API connection verified")
except Exception as e:
    print(f"API error: {e}")

## Stage 1 — Load the corpus

The corpus consists of 66 publicly available UK public dialogue reports. PDFs are downloaded
automatically from a public Google Drive folder; metadata (filename → technology category, year)
is loaded from a public Google Sheet.

Each PDF is split into paragraphs using double-newline boundaries. Paragraphs shorter than
the minimum word threshold (`min_chunk_words`, default 50) are discarded — they are typically
table-of-contents lines, page footers, or headings that carry no substantive participant voice.

The data quality cell reports the distribution of paragraph lengths and flags any documents that
produced unusually few chunks.

The chunked corpus is cached to `chunks_df.parquet` in the checkpoint folder so subsequent runs skip the PDF processing entirely.

In [ ]:
# @title Upload source PDF documents
show_status("Preparing PDF upload...")

from google.colab import files

print("Upload your PDF files:")
uploaded = files.upload()

pdf_files = []
for filename, content in uploaded.items():
    if filename.endswith('.pdf'):
        filepath = PDF_FOLDER / filename
        filepath.write_bytes(content)
        pdf_files.append(filepath)

show_complete(f"Uploaded {len(pdf_files)} PDF files")

In [ ]:
# @title Upload document metadata
show_status("Upload metadata file...")

print("Upload Excel file with columns: filename, technology, year")
uploaded = files.upload()

metadata_df = None
for fn, content in uploaded.items():
    if fn.endswith(('.xlsx', '.xls')):
        path = OUTPUT_FOLDER / fn
        path.write_bytes(content)
        metadata_df = pd.read_excel(path)
        break

if metadata_df is None:
    raise ValueError("No Excel file uploaded!")

required = ['filename', 'technology', 'year']
missing = [c for c in required if c not in metadata_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

metadata_lookup = metadata_df.set_index('filename').to_dict('index')

print(f"\nTechnology Distribution:")
print(metadata_df['technology'].value_counts().to_string())
print(f"\nYear Range: {metadata_df['year'].min()} - {metadata_df['year'].max()}")

show_complete(f"Metadata loaded for {len(metadata_df)} documents")

In [ ]:
# @title Extract paragraphs from PDFs (chunk the corpus)

show_status("Extracting paragraphs from PDFs...")

_chunks_cache = CHECKPOINT_FOLDER / "chunks_df.parquet"

if _chunks_cache.exists():
    chunks_df = pd.read_parquet(_chunks_cache)
    show_complete(f"Loaded {len(chunks_df):,} chunks from cache ({_chunks_cache})")
else:
    all_chunks = []
    meta_lookup = {}
    if "metadata_df" in globals():
        for _, row in metadata_df.iterrows():
            meta_lookup[row["filename"]] = {
                "technology": row.get("technology", "Unknown"),
                "year": row.get("year", None),
            }

    for pdf_path in sorted(pdf_files):
        pdf_meta = meta_lookup.get(pdf_path.name, {"technology": "Unknown", "year": None})
        chunks = extract_chunks_from_pdf(
            str(pdf_path),
            pdf_meta,
            min_chunk_words=MIN_CHUNK_WORDS,
            max_chunk_words=MAX_CHUNK_WORDS,
            min_chunk_chars=MIN_CHUNK_CHARS,
        )
        all_chunks.extend(chunks)

    chunks_df = pd.DataFrame(all_chunks)
    # Assign a stable chunk_id
    chunks_df["chunk_id"] = [
        f"{row['source_file']}__{row['chunk_index']}"
        for _, row in chunks_df.iterrows()
    ]
    # Ensure technology_meta column exists
    if "technology_meta" not in chunks_df.columns:
        chunks_df["technology_meta"] = chunks_df["technology"]

    # Save to cache
    chunks_df.to_parquet(_chunks_cache, index=False)
    show_complete(
        f"Extracted {len(chunks_df):,} chunks from {len(pdf_files)} PDFs "
        f"→ cached to {_chunks_cache}"
    )

print(f"  Technologies: {sorted(chunks_df['technology_meta'].unique())}")
print(f"  Year range:   {chunks_df['year'].min()} – {chunks_df['year'].max()}")
print(f"  Median words: {chunks_df['word_count'].median():.0f}")

In [ ]:
# @title Summarise paragraph-level data quality

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Chunks by technology
tech_counts = chunks_df['technology_meta'].value_counts()
axes[0, 0].barh(tech_counts.index, tech_counts.values, color='steelblue')
axes[0, 0].set_xlabel('Number of Paragraphs')
axes[0, 0].set_title('Paragraphs by Technology')
axes[0, 0].invert_yaxis()

# Chunks by year
year_counts = chunks_df.groupby('year').size()
axes[0, 1].bar(year_counts.index.astype(str), year_counts.values, color='steelblue')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Paragraphs')
axes[0, 1].set_title('Paragraphs by Year')
axes[0, 1].tick_params(axis='x', rotation=45)

# Word count distribution
axes[1, 0].hist(chunks_df['word_count'], bins=30, color='steelblue', edgecolor='white')
axes[1, 0].set_xlabel('Word Count')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Paragraph Length Distribution')

# Chunks per document
doc_chunks = chunks_df.groupby('source_file').size().sort_values(ascending=False)
axes[1, 1].bar(range(len(doc_chunks)), doc_chunks.values, color='steelblue')
axes[1, 1].set_xlabel('Document Index')
axes[1, 1].set_ylabel('Paragraphs')
axes[1, 1].set_title('Paragraphs per Document')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "data_quality_overview.png", dpi=150)
plt.show()

## Stage 2 — LLM phrase extraction (concerns)

Each paragraph is sent to the LLM with a structured prompt asking it to extract
*decontextualised* concern phrases — short, self-contained statements that make sense without
the surrounding text. The prompt includes explicit negative constraints to prevent
meta-vocabulary artefacts (e.g. the word "public dialogue" should never appear in a phrase).

**Cache behaviour:** Extracted phrases are cached to `extracted_concerns.json`. On subsequent runs
the cache is reused if the chunk IDs match *and* it passes a validity check (fewer than 30% of
entries are empty — a sign of a partial-failure run).

**Outputs:**
- `concerns_df` — one row per extracted phrase with source document, technology, and year
- `extraction_yield_summary.csv` — per-document yield statistics
- `tech_filter_drops_concern.csv` — phrases dropped by the technology-word filter
- `extraction_errors_concern.csv` — chunks where the LLM returned an error

In [ ]:
# @title Extract decontextualised concern phrases

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

show_status("Extracting decontextualised concern phrases...")

# CONCERN_PROMPT is defined in dialogue_utils (includes anti-artefact rules)
# extract_concerns_from_paragraph replaced by extract_phrases(row, kind='concern', ...) from dialogue_utils

# Check for cached extractions
concerns_cache_file = CHECKPOINT_FOLDER / "extracted_concerns.json"

if concerns_cache_file.exists():
    print("Found cached concern extractions...")

    with open(concerns_cache_file) as f:
        cached_concerns = json.load(f)

    cached_ids = set(cached_concerns.keys())
    current_ids = set(chunks_df['chunk_id'].tolist())

    if cached_ids == current_ids:
        print(f"Using cached extractions for {len(cached_concerns)} paragraphs")

        if validate_extraction_cache(cached_concerns, "concern"):
            all_concerns = cached_concerns
        else:
            print("Cache validation failed — will re-extract")
            all_concerns = None
    else:
        print("Cache mismatch — will re-extract")
        all_concerns = None

else:
    all_concerns = None


# Run extraction if no valid cache
if all_concerns is None:

    all_concerns = {}

    # Build lookup for metadata
    chunk_metadata = (
        chunks_df
        .set_index('chunk_id')[['technology', 'year', 'source_file']]
        .to_dict('index')
    )

    MAX_WORKERS = 10
    rows = list(chunks_df.iterrows())

    concern_results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

        futures = {
            executor.submit(
                extract_phrases,
                row,
                'concern',
                client,
                None,
                LLM_MODEL
            ): row[1]['chunk_id']
            for row in rows
        }

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Extracting concerns"
        ):

            result = future.result()
            concern_results.append(result)

            chunk_id = result.chunk_id
            concerns = result.retained_phrases

            meta = chunk_metadata[chunk_id]

            all_concerns[chunk_id] = {
                'concerns': concerns,
                'technology': meta['technology'],
                'year': int(meta['year']) if pd.notna(meta['year']) else None,
                'source_file': meta['source_file']
            }

    # Cache results
    with open(concerns_cache_file, 'w') as f:
        json.dump(all_concerns, f, indent=2)

    # Write diagnostics
    write_extraction_diagnostics(
        concern_results,
        kind='concern',
        output_folder=OUTPUT_FOLDER
    )

    show_complete(
        f"Extracted concerns from {len(all_concerns)} paragraphs"
    )


# Flatten to individual concern rows
concern_rows = []

for chunk_id, data in all_concerns.items():

    for concern in data['concerns']:

        concern_rows.append({
            'chunk_id': chunk_id,
            'concern': concern,
            'technology': data['technology'],
            'year': data['year'],
            'source_file': data['source_file']
        })

concerns_df = pd.DataFrame(concern_rows)

concerns_df['concern_id'] = [
    f"concern_{i}" for i in range(len(concerns_df))
]

print("\nExtraction Summary:")
print(f"  Paragraphs processed: {len(all_concerns)}")
print(f"  Total concern phrases: {len(concerns_df)}")
print(
    f"  Avg concerns per paragraph: "
    f"{len(concerns_df) / len(all_concerns):.2f}"
)
print(
    f"  Paragraphs with no concerns: "
    f"{sum(1 for d in all_concerns.values() if len(d['concerns']) == 0)}"
)

concerns_df.to_csv(
    OUTPUT_FOLDER / "extracted_concerns.csv",
    index=False
)

In [ ]:
# Make sure concerns_df knows the technology of each paragraph
concerns_df = concerns_df.merge(
    chunks_df[["chunk_id", "technology_meta"]],
    on="chunk_id",
    how="left"
)


In [ ]:
# @title Inspect sample concern extractions

print("Sample extracted concerns by technology:")
print("(Checking that technology-specific language has been removed)\n")

for tech in concerns_df['technology_meta'].unique()[:6]:
    tech_concerns = concerns_df[concerns_df['technology_meta'] == tech]['concern'].head(8).tolist()
    print(f"\n{tech}:")
    for c in tech_concerns:
        print(f"  • {c}")

In [ ]:
# @title Vocabulary frequency diagnostic (concerns) — CIP-0004
# Flags meta-vocabulary terms (e.g. "public dialogue") that may be
# prompt artefacts rather than genuine public concerns.

vocab_concern_df = vocabulary_frequency_diagnostic(
    phrases=concerns_df["concern"].tolist(),
    kind="concern",
    output_folder=OUTPUT_FOLDER,
)

# Quick view of any flagged meta-vocabulary in the top 100 terms
_flagged = vocab_concern_df[vocab_concern_df["is_meta_vocab"]]
if not _flagged.empty:
    flagged_str = ", ".join(
        f'"{r["term"]}": {r["pct_of_phrases"]}%' for _, r in _flagged.iterrows()
    )
    show_warning(f"Meta-vocabulary detected in concern phrases: {flagged_str}")
else:
    show_complete("No meta-vocabulary terms in top-100 concern terms.")

vocab_concern_df.head(20)


## Stage 3 — Semantic embeddings and clustering (concerns)

Each concern phrase is converted to a 1536-dimensional embedding using
`text-embedding-3-small`. Embeddings are L2-normalised and stored to cache.

The embeddings are then clustered using k-means (`N_CONCERN_CLUSTERS`). Each cluster
represents a thematic group of related concerns. After clustering:

- **Technology entropy** is computed for each cluster — a cluster with high entropy
  spans many technologies (cross-cutting); low entropy means it concentrates in one or two.
- The threshold `CROSSCUTTING_ENTROPY_THRESHOLD = 0.5` is used to classify clusters.
- **Vocabulary diagnostic** (`concern_vocab_frequency.csv`) shows the top unigrams and bigrams
  to help spot any remaining meta-vocabulary artefacts.

In [ ]:
# @title Generate semantic embeddings for concern phrases

show_status(f"Generating embeddings for {len(concerns_df)} concern phrases...")

_emb_file  = CHECKPOINT_FOLDER / "concern_embeddings.npy"
_ids_file  = CHECKPOINT_FOLDER / "concern_ids.json"

if _emb_file.exists() and _ids_file.exists():
    embeddings = np.load(_emb_file)
    with open(_ids_file) as _f:
        _cached_ids = json.load(_f)
    if _cached_ids == concerns_df["concern_id"].tolist():
        show_complete(f"Loaded cached concern embeddings: {embeddings.shape}")
    else:
        show_warning("Embedding cache mismatch (IDs differ) — regenerating")
        embeddings = None
else:
    embeddings = None

if embeddings is None:
    # Filter out any rows with empty concern text before embedding
    concerns_df = filter_missing_source_text(concerns_df, text_col="concern")

    texts = concerns_df["concern"].tolist()
    all_embeddings = []

    for _i in tqdm(range(0, len(texts), EMBEDDING_BATCH_SIZE), desc="Embedding concerns"):
        batch = texts[_i : _i + EMBEDDING_BATCH_SIZE]
        try:
            batch_embeddings = get_embeddings_batch(batch, client, model=EMBEDDING_MODEL)
            all_embeddings.append(batch_embeddings)
        except Exception as _e:
            print(f"Error on batch {_i}: {_e}")
        time.sleep(0.1)

    embeddings = np.vstack(all_embeddings) if all_embeddings else np.empty((0, 0))
    np.save(_emb_file, embeddings)
    with open(_ids_file, "w") as _f:
        json.dump(concerns_df["concern_id"].tolist(), _f)
    show_complete(f"Generated concern embeddings: {embeddings.shape} → cached")

print(f"  Embedding dimensions: {embeddings.shape[1] if embeddings.ndim > 1 else 'n/a'}")

In [ ]:
# @title Cluster concern phrase embeddings

show_status(f"Clustering into {N_CONCERN_CLUSTERS} concern groups...")

embeddings_normalized = normalize(embeddings)

kmeans = KMeans(
    n_clusters=N_CONCERN_CLUSTERS,
    random_state=RANDOM_SEED,
    n_init=10,
    max_iter=300
)
cluster_labels = kmeans.fit_predict(embeddings_normalized)

centroids = kmeans.cluster_centers_
centroids_normalized = normalize(centroids)

# Add cluster assignment to concerns dataframe
concerns_df['cluster_id'] = cluster_labels

# SOFT MEMBERSHIP: cosine similarity to each centroid
show_status("Computing soft membership scores...")
soft_membership = cosine_similarity(embeddings_normalized, centroids_normalized)

print(f"\nClustering Results:")
print(f"  Clusters: {N_CONCERN_CLUSTERS}")
print(f"  Concern phrases: {len(concerns_df)}")
print(f"  Soft membership matrix: {soft_membership.shape}")

cluster_sizes = np.bincount(cluster_labels)
print(f"  Cluster sizes: min={cluster_sizes.min()}, max={cluster_sizes.max()}, median={np.median(cluster_sizes):.0f}")

np.save(CHECKPOINT_FOLDER / "soft_membership.npy", soft_membership)
np.save(CHECKPOINT_FOLDER / "cluster_centroids.npy", centroids_normalized)

show_complete("Clustering complete")

In [ ]:
# @title Characterise clusters by technology distribution

show_status("Analysing cluster composition...")

# For each cluster, compute entropy of technology distribution
cluster_entropy = {}
cluster_tech_dist = {}

technologies = concerns_df['technology_meta'].unique()
n_techs = len(technologies)

for cluster_id in range(N_CONCERN_CLUSTERS):
    cluster_mask = concerns_df['cluster_id'] == cluster_id
    cluster_data = concerns_df[cluster_mask]

    if len(cluster_data) == 0:
        cluster_entropy[cluster_id] = 0
        cluster_tech_dist[cluster_id] = {}
        continue

    # Technology distribution
    tech_counts = cluster_data['technology_meta'].value_counts()
    tech_probs = tech_counts / tech_counts.sum()

    # Entropy (higher = more cross-cutting)
    cluster_entropy[cluster_id] = entropy(tech_probs)
    cluster_tech_dist[cluster_id] = tech_counts.to_dict()

# Normalize entropy to 0-1 scale (max entropy = log(n_techs))
max_entropy = np.log(n_techs)
normalized_entropy = {k: v / max_entropy for k, v in cluster_entropy.items()}

# Classify clusters
CROSS_CUTTING_THRESHOLD = CROSSCUTTING_ENTROPY_THRESHOLD  # from dialogue_utils

cross_cutting_clusters = [k for k, v in normalized_entropy.items() if v >= CROSS_CUTTING_THRESHOLD]
tech_specific_clusters = [k for k, v in normalized_entropy.items() if v < CROSS_CUTTING_THRESHOLD]

print(f"\nCluster Classification:")
print(f"  Cross-cutting clusters (entropy >= {CROSS_CUTTING_THRESHOLD}): {len(cross_cutting_clusters)}")
print(f"  Technology-specific clusters: {len(tech_specific_clusters)}")
print(f"  Ratio: {len(cross_cutting_clusters)/N_CONCERN_CLUSTERS*100:.1f}% cross-cutting")

# Visualise entropy distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of entropy
axes[0].hist(list(normalized_entropy.values()), bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({CROSS_CUTTING_THRESHOLD})')
axes[0].set_xlabel('Normalized Entropy')
axes[0].set_ylabel('Number of Clusters')
axes[0].set_title('Cluster Entropy Distribution\n(Higher = More Cross-Cutting)')
axes[0].legend()

# Entropy vs cluster size
sizes = [cluster_sizes[i] for i in range(N_CONCERN_CLUSTERS)]
entropies = [normalized_entropy[i] for i in range(N_CONCERN_CLUSTERS)]
colors = ['green' if e >= CROSS_CUTTING_THRESHOLD else 'orange' for e in entropies]
axes[1].scatter(sizes, entropies, c=colors, alpha=0.6)
axes[1].axhline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--')
axes[1].set_xlabel('Cluster Size')
axes[1].set_ylabel('Normalized Entropy')
axes[1].set_title('Cross-Cutting (green) vs Technology-Specific (orange)')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "cluster_entropy_analysis.png", dpi=150)
plt.show()

show_complete("Cluster composition analysis complete")

In [ ]:
# --- Define cross-cutting helpers for Section 6 (missing in v10-2) ---

# Use the same threshold already used in Section 3
CROSS_CUTTING_ENTROPY_THRESHOLD = CROSS_CUTTING_THRESHOLD  # already 0.5 in the notebook

# IDs of cross-cutting clusters
cross_cutting_cluster_ids = set(cross_cutting_clusters)

# Optional: map cluster_id -> human label if we have GPT labels (summary_df)
label_map = {}
if "summary_df" in globals() and {"cluster_id", "label"}.issubset(summary_df.columns):
    label_map = dict(zip(summary_df["cluster_id"], summary_df["label"]))

# Provide the dict Section 6.2 expects
cross_cutting_labels = {cid: label_map.get(cid, f"Cluster {cid}") for cid in cross_cutting_cluster_ids}


## Stage 4 — Cluster labelling (concerns)

For each cluster, the 8 phrases closest to the cluster centroid (the "exemplars") are sent
to the LLM, which returns:
- A **short label** (3–6 words)
- A **one-sentence description**
- A list of **key terms**

The LLM sees only the phrase text — no technology names or document metadata are included in
the prompt, so labels reflect the content of participants' concerns rather than which technology
prompted them.

In [ ]:
# @title Extract exemplar concern phrases per cluster

show_status("Extracting exemplar concerns...")

N_EXEMPLARS = 8
cluster_exemplars = {}

for cluster_id in range(N_CONCERN_CLUSTERS):
    cluster_mask = concerns_df['cluster_id'] == cluster_id
    cluster_concerns = concerns_df[cluster_mask]
    cluster_embs = embeddings_normalized[cluster_mask]

    if len(cluster_concerns) == 0:
        continue

    centroid = centroids_normalized[cluster_id]
    similarities = cosine_similarity(cluster_embs, centroid.reshape(1, -1)).flatten()
    top_indices = np.argsort(similarities)[-N_EXEMPLARS:][::-1]

    exemplars = []
    for idx in top_indices:
        row = cluster_concerns.iloc[idx]
        exemplars.append({
            'concern': row['concern'],
            'technology': row['technology'],
            'year': int(row['year']) if pd.notna(row['year']) else None,
            'similarity': float(similarities[idx])
        })

    # Cluster metadata
    tech_dist = cluster_concerns['technology'].value_counts().head(3).to_dict()

    cluster_exemplars[cluster_id] = {
        'size': len(cluster_concerns),
        'entropy': normalized_entropy[cluster_id],
        'is_cross_cutting': cluster_id in cross_cutting_clusters,
        'top_technologies': tech_dist,
        'exemplars': exemplars
    }

with open(OUTPUT_FOLDER / "cluster_exemplars.json", 'w') as f:
    json.dump(cluster_exemplars, f, indent=2, default=str)

show_complete(f"Extracted exemplars for {len(cluster_exemplars)} clusters")

In [ ]:
# label_cluster (concerns) — imported from dialogue_utils


---
# SECTION 5: Framing lens analysis
---

In [ ]:
# @title Label concern clusters

show_status("Labelling concern clusters...")

labels_file = OUTPUT_FOLDER / "cluster_labels.json"

if labels_file.exists():
    print("Found cached cluster labels...")
    with open(labels_file) as f:
        cluster_labels_dict = json.load(f)

else:
    cluster_labels_dict = {}

    for cluster_id in tqdm(cluster_exemplars.keys(), desc="Labelling clusters"):

        data = cluster_exemplars[cluster_id]

        result = label_cluster(
            cluster_id,
            data["exemplars"],
            data["is_cross_cutting"]
        )

        cluster_labels_dict[str(cluster_id)] = result

    with open(labels_file, "w") as f:
        json.dump(cluster_labels_dict, f, indent=2)

show_complete(f"Labelled {len(cluster_labels_dict)} concern clusters")

print("\nCluster labels:")
for cluster_id, label_data in cluster_labels_dict.items():
    print(f"{cluster_id}. {label_data.get('label', f'Cluster {cluster_id}')}")

In [ ]:
# @title Build ordered label list for downstream analysis

cluster_labels_list = []

for cluster_id in sorted(cluster_exemplars.keys()):

    label_data = cluster_labels_dict.get(str(cluster_id), {})

    cluster_labels_list.append(
        label_data.get("label", f"Cluster {cluster_id}")
    )

In [ ]:
# @title Build readable cluster label lookup

CLUSTER_LABELS = {}

for cluster_id, label_data in cluster_labels_dict.items():
    CLUSTER_LABELS[int(cluster_id)] = label_data.get(
        "label",
        f"Cluster {cluster_id}"
    )

print(f"Built readable labels for {len(CLUSTER_LABELS)} clusters.")

In [ ]:
# @title Identify framing lenses

show_status("Generating framing lens suggestions...")

# Prepare cluster summary for GPT
cluster_info = []
for cluster_id, data in cluster_exemplars.items():
    label = cluster_labels_dict.get(str(cluster_id), {}).get('label', f'Cluster {cluster_id}')
    description = cluster_labels_dict.get(str(cluster_id), {}).get('description', '')
    cluster_type = 'cross-cutting' if data['is_cross_cutting'] else 'tech-specific'
    cluster_info.append(f"{cluster_id}. {label} ({cluster_type}, n={data['size']}): {description}")

cluster_summary_text = "\n".join(cluster_info)

lens_prompt = f"""Analyze these {N_CONCERN_CLUSTERS} concern clusters from UK public dialogue reports.

Clusters:
{cluster_summary_text}

Group these clusters into 8-12 higher-level FRAMING LENSES that capture how publics frame their concerns.

For each lens provide:
1. Name (2-4 words)
2. Description (1 sentence)
3. List of cluster IDs that belong to this lens

A cluster can belong to multiple lenses if appropriate.
Ensure all clusters are assigned to at least one lens.

Return JSON:
{{"framing_lenses": [
  {{"name": "...", "description": "...", "suggested_clusters": [0, 1, 2...]}},
  ...
]}}"""

try:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "Expert in public engagement and discourse analysis. Return only valid JSON."},
            {"role": "user", "content": lens_prompt}
        ],
        max_completion_tokens=8000
    )

    content = response.choices[0].message.content
    if '```' in content:
        parts = content.split('```')
        for part in parts:
            if part.startswith('json'):
                content = part[4:].strip()
                break
            elif part.strip().startswith('{'):
                content = part.strip()
                break

    suggested_lenses = json.loads(content)

    print("Suggested Framing Lenses:\n")
    for lens in suggested_lenses['framing_lenses']:
        n_clusters = len(lens['suggested_clusters'])
        print(f"  {lens['name']}")
        print(f"    {lens['description']}")
        print(f"    Clusters: {lens['suggested_clusters'][:8]}...\n")

    show_complete(f"Generated {len(suggested_lenses['framing_lenses'])} framing lenses")

except Exception as e:
    print(f"Error generating lenses: {e}")
    suggested_lenses = {'framing_lenses': []}

In [ ]:
# @title Map concern clusters to framing lenses

FRAMING_LENS_MAPPINGS = {}
for lens in suggested_lenses['framing_lenses']:
    FRAMING_LENS_MAPPINGS[lens['name']] = {
        'description': lens['description'],
        'cluster_ids': lens['suggested_clusters']
    }

with open(OUTPUT_FOLDER / "framing_lens_mappings.json", 'w') as f:
    json.dump(FRAMING_LENS_MAPPINGS, f, indent=2)

# Map clusters to lenses
cluster_to_lens = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    for cluster_id in data['cluster_ids']:
        if cluster_id not in cluster_to_lens:
            cluster_to_lens[cluster_id] = []
        cluster_to_lens[cluster_id].append(lens_name)

print("Framing Lens Mappings:")
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    n_clusters = len(data['cluster_ids'])
    # Count concerns in this lens
    lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
    n_concerns = lens_mask.sum()
    print(f"  {lens_name}: {n_clusters} clusters, {n_concerns} concerns")

In [ ]:
# @title Compute framing lens prevalence

show_status("Computing framing lens prevalence...")

lens_prevalence = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
    lens_prevalence[lens_name] = lens_mask.sum()

lens_prev_df = pd.DataFrame([
    {'Framing Lens': k, 'Concerns': v}
    for k, v in lens_prevalence.items()
]).sort_values('Concerns', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(lens_prev_df['Framing Lens'], lens_prev_df['Concerns'], color='steelblue')
ax.set_xlabel('Number of Concern Phrases')
ax.set_title('Framing Lens Prevalence Across All Documents')

for bar, val in zip(bars, lens_prev_df['Concerns']):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "framing_lens_prevalence.png", dpi=150, bbox_inches='tight')
plt.show()

show_complete("Framing lens prevalence computed")

## Stage 5 — Cross-cutting analysis and AI distinctiveness (concerns)

This stage asks two questions:

1. **What concerns are cross-cutting?** Clusters with normalised technology entropy ≥ 0.5
   appear across many technologies, suggesting they reflect general public attitudes to novel
   technologies rather than technology-specific reactions.

2. **What is distinctive about AI dialogues?** For each concern cluster, we compute the
   fraction of AI-domain documents that mention it vs. the fraction of non-AI documents.
   Clusters where AI is substantially over-represented are "AI-distinctive" concerns.

The temporal analysis (further below) tracks how concern cluster prevalence shifts by year,
normalised by the number of dialogues conducted that year.

In [ ]:
# --- Section 6 prerequisites (canonical tech + cross-cutting clusters) ---

TECH_COL = "technology_meta"

# Ensure concerns_df has technology_meta
if TECH_COL not in concerns_df.columns:
    concerns_df = concerns_df.merge(
        chunks_df[["chunk_id", TECH_COL]],
        on="chunk_id",
        how="left"
    )

# Define cross-cutting clusters from entropy (single source of truth)
CROSS_CUTTING_ENTROPY_THRESHOLD = 0.5
cross_cutting_cluster_ids = {
    cid for cid, ent in cluster_entropy.items()
    if ent >= CROSS_CUTTING_ENTROPY_THRESHOLD
}


In [ ]:
# @title Shared concerns across technologies (document-weighted)

show_status("Computing shared concern structure (all technologies)...")

TECH_COL = "technology_meta"

# Canonical list of technologies from metadata
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())

# Salience matrix: rows=technology, cols=cluster_id, values=proportion of concerns
salience_rows = {}
for tech in technologies:
    tech_mask = concerns_df[TECH_COL] == tech
    tech_total = int(tech_mask.sum())
    if tech_total == 0:
        continue
    counts = concerns_df.loc[tech_mask, "cluster_id"].value_counts()
    salience_rows[tech] = (counts / tech_total).to_dict()

salience_df = pd.DataFrame(salience_rows).T.fillna(0)

# Document-weighted global prevalence of clusters
global_cluster_prevalence = concerns_df["cluster_id"].value_counts(normalize=True)

# Cross-cutting metric (entropy over technologies) computed earlier as cluster_entropy
cluster_meta_df = (
    pd.DataFrame(
        {
            "cluster_id": list(cluster_entropy.keys()),
            "tech_entropy": [cluster_entropy[c] for c in cluster_entropy.keys()],
            "global_prevalence": [global_cluster_prevalence.get(c, 0) for c in cluster_entropy.keys()],
        }
    )
    .set_index("cluster_id")
    .sort_values(["global_prevalence", "tech_entropy"], ascending=False)
)

# Shared concerns = high prevalence + high entropy
shared_concerns = cluster_meta_df.head(20)

print("Top shared concern clusters (high prevalence + cross-technology spread):")
display(shared_concerns.head(12))

# Quick bar chart (prevalence)
plt.figure(figsize=(10, 5))
shared_concerns.sort_values("global_prevalence")["global_prevalence"].plot(kind="barh", legend=False)
plt.title("Shared concerns across technologies (top 20 clusters by prevalence × spread)")
plt.xlabel("Share of all extracted concern phrases")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_concerns_top20.png")
plt.show()

shared_concerns.reset_index().to_csv(OUTPUT_FOLDER / "shared_concerns_top20.csv", index=False)

show_complete("Shared concerns computed")


In [ ]:
# @title Shared framings across technologies (document-weighted)

show_status("Computing shared framing structure (all technologies)...")

TECH_COL = 'technology_meta'
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())

# Lens prevalence overall + entropy across technologies
lens_stats = []
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_clusters = set(data['cluster_ids'])
    lens_mask = concerns_df['cluster_id'].isin(lens_clusters)
    overall_prev = lens_mask.mean()  # document-weighted (each concern phrase counts equally)

    # Technology distribution within lens
    tech_counts = concerns_df.loc[lens_mask, TECH_COL].value_counts()
    tech_probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum() > 0 else np.array([])
    lens_entropy = float(entropy(tech_probs)) if len(tech_probs) > 1 else 0.0

    lens_stats.append({
        'framing_lens': lens_name,
        'overall_prevalence': float(overall_prev),
        'tech_entropy': lens_entropy,
        'n_concerns_in_lens': int(lens_mask.sum())
    })

lens_meta_df = (pd.DataFrame(lens_stats)
                .sort_values(['overall_prevalence','tech_entropy'], ascending=False))

print("Shared framings (high prevalence + cross-technology spread):")
display(lens_meta_df.head(12))

# Scatter: prevalence vs entropy
plt.figure(figsize=(7,5))
plt.scatter(lens_meta_df['tech_entropy'], lens_meta_df['overall_prevalence'])
for _, r in lens_meta_df.iterrows():
    plt.text(r['tech_entropy'], r['overall_prevalence'], r['framing_lens'], fontsize=8)
plt.xlabel("Entropy across technologies")
plt.ylabel("Share of all extracted concerns")
plt.title("Shared framings across technologies")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_framings_scatter.png")
plt.show()

lens_meta_df.to_csv(OUTPUT_FOLDER / "shared_framings.csv", index=False)
show_complete("Shared framings computed")


In [ ]:
# @title Compare AI and non-AI dialogues by framing lens

show_status("Computing AI vs non-AI framing profile (technology-weighted baseline)...")

TECH_COL = 'technology_meta'
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Lens salience per technology (within-technology proportions)
lens_salience_by_tech = {}
for tech in technologies:
    tech_mask = concerns_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    lens_salience_by_tech[tech] = {}
    for lens_name, data in FRAMING_LENS_MAPPINGS.items():
        lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
        lens_salience_by_tech[tech][lens_name] = float((tech_mask & lens_mask).sum() / tech_total)

lens_salience_df = pd.DataFrame(lens_salience_by_tech).T.fillna(0)

# Technology-weighted non-AI baseline (each non-AI technology counts equally)
non_ai_avg = lens_salience_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

# Plot radar: AI vs non-AI avg
if 'AI' in lens_salience_df.index and len(non_ai_avg) > 0:
    categories = list(non_ai_avg.index)
    ai_vals = lens_salience_df.loc['AI', categories].tolist()
    base_vals = non_ai_avg[categories].tolist()

    # Close the loop
    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI framing profile vs non‑AI baseline (technology‑weighted)"
    )
    fig.write_html(OUTPUT_FOLDER / "ai_vs_nonai_framing_radar.html")
    fig.show()

# Save table
lens_salience_df.to_csv(OUTPUT_FOLDER / "lens_salience_by_technology_meta.csv")
non_ai_avg.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "non_ai_avg_framing_profile.csv")
show_complete("AI vs non-AI framing profile computed")


In [ ]:
# @title Compare AI and non-AI dialogues on cross-cutting concerns

show_status("Computing AI vs non-AI concern profile on cross-cutting clusters...")

TECH_COL = 'technology_meta'
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Use cross_cutting_labels from SECTION 3 (derived from entropy threshold)
cross_cutting_cluster_ids = set(cross_cutting_labels.keys())

# Technology profiles over cross-cutting clusters
profiles = {}
for tech in technologies:
    tech_mask = concerns_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    counts = concerns_df.loc[tech_mask & concerns_df['cluster_id'].isin(cross_cutting_cluster_ids), 'cluster_id'].value_counts()
    # normalise over all concerns for that tech (so "attention share" to cross-cutting clusters)
    profiles[tech] = (counts / tech_total).to_dict()

profiles_df = pd.DataFrame(profiles).T.fillna(0)

non_ai_avg = profiles_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

# Save for later use
profiles_df.to_csv(OUTPUT_FOLDER / "cross_cutting_cluster_profiles_by_tech.csv")
non_ai_avg.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "cross_cutting_cluster_profile_non_ai_avg.csv")

show_complete("Cross-cutting concern profiles computed")


In [ ]:
# --- Build CLUSTER_LABELS dict for plots (cluster_id -> label) ---
if "summary_df" in globals() and {"cluster_id", "label"}.issubset(summary_df.columns):
    CLUSTER_LABELS = dict(zip(summary_df["cluster_id"], summary_df["label"]))
else:
    CLUSTER_LABELS = {}


In [ ]:
# @title Visualise AI concern fingerprint (AI vs non-AI)

TECH_COL = 'technology_meta'

if 'AI' in profiles_df.index and len(non_ai_avg) > 0:
    show_status("Creating AI fingerprint radar (AI vs non-AI)...")

    diff = (profiles_df.loc['AI'] - non_ai_avg).sort_values()
    top_low = diff.head(5).index.tolist()
    top_high = diff.tail(7).index.tolist()
    selected = top_low + top_high

    # Human-friendly labels
    def pretty_cluster(cid):
        return CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    categories = [pretty_cluster(c) for c in selected]
    ai_vals = profiles_df.loc['AI', selected].tolist()
    base_vals = non_ai_avg[selected].tolist()

    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI concern fingerprint vs non‑AI baseline (cross‑cutting clusters)"
    )
    fig.write_html(OUTPUT_FOLDER / "ai_vs_nonai_concern_radar.html")
    fig.show()

show_complete("AI fingerprint radar created")


In [ ]:
# @title Identify distinctive AI concerns and framings

show_status("Computing distinctive concerns and framings for AI...")

TECH_COL = 'technology_meta'
non_ai_techs = [t for t in technologies if t != 'AI']

# Distinctive concern clusters (all clusters, tech-weighted baseline)
all_cluster_profiles = salience_df  # from 6.1a (rows=tech, cols=cluster_id)

if 'AI' in all_cluster_profiles.index and len(non_ai_techs) > 0:
    non_ai_avg_all = all_cluster_profiles.loc[non_ai_techs].mean(axis=0)
    ai_vs_base = (all_cluster_profiles.loc['AI'] - non_ai_avg_all).sort_values()

    most_under = ai_vs_base.head(10)
    most_over = ai_vs_base.tail(10)

    def label(cid):
        return CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    distinctive_concerns = pd.DataFrame({
        'cluster_id': list(most_under.index) + list(most_over.index),
        'cluster_label': [label(c) for c in list(most_under.index) + list(most_over.index)],
        'ai_minus_nonai': list(most_under.values) + list(most_over.values),
        'ai_share': [float(all_cluster_profiles.loc['AI', c]) for c in list(most_under.index) + list(most_over.index)],
        'nonai_share': [float(non_ai_avg_all[c]) for c in list(most_under.index) + list(most_over.index)],
    }).sort_values('ai_minus_nonai')

    display(distinctive_concerns)
    distinctive_concerns.to_csv(OUTPUT_FOLDER / "ai_distinctive_concerns.csv", index=False)

# Distinctive framings (lens level)
if 'AI' in lens_salience_df.index and len(non_ai_techs) > 0:
    non_ai_avg_lens = lens_salience_df.loc[non_ai_techs].mean(axis=0)
    lens_diff = (lens_salience_df.loc['AI'] - non_ai_avg_lens).sort_values()

    df = pd.DataFrame({
        'framing_lens': lens_diff.index,
        'ai_minus_nonai': lens_diff.values,
        'ai_share': lens_salience_df.loc['AI', lens_diff.index].values,
        'nonai_share': non_ai_avg_lens[lens_diff.index].values
    }).sort_values('ai_minus_nonai')

    display(df.head(10))
    display(df.tail(10))
    df.to_csv(OUTPUT_FOLDER / "ai_distinctive_framings.csv", index=False)

show_complete("Distinctiveness outputs saved")


In [ ]:
# @title Analyse AI-specific concerns

show_status("Identifying AI-specific concerns...")

# Find clusters that are predominantly AI
ai_specific_clusters = []

for cluster_id in tech_specific_clusters:
    tech_dist = cluster_tech_dist.get(cluster_id, {})
    total = sum(tech_dist.values())
    if total == 0:
        continue

    ai_count = tech_dist.get('AI', 0)
    ai_share = ai_count / total

    if ai_share > 0.4:
        ai_specific_clusters.append({
            'cluster_id': cluster_id,
            'label': cluster_labels_list[cluster_id],
            'size': int(cluster_sizes[cluster_id]),
            'ai_share': ai_share,
            'ai_count': ai_count
        })

# Collect AI presence in all tech-specific clusters
ai_in_tech_specific = []
for cluster_id in tech_specific_clusters:
    tech_dist = cluster_tech_dist.get(cluster_id, {})
    ai_count = tech_dist.get('AI', 0)
    total = sum(tech_dist.values())
    ai_in_tech_specific.append({
        'cluster_id': cluster_id,
        'label': cluster_labels_list[cluster_id],
        'size': int(cluster_sizes[cluster_id]),
        'ai_count': ai_count,
        'ai_share': ai_count / total if total > 0 else 0
    })

ai_in_tech_df = pd.DataFrame(ai_in_tech_specific).sort_values('ai_count', ascending=False)

print("="*70)
print("KEY FINDING: AI Concerns Are Not Distinctive at the Cluster Level")
print("="*70)

print(f"\nOf {len(tech_specific_clusters)} technology-specific clusters:")
print(f"  • Clusters where AI > 50%: {len([c for c in ai_specific_clusters if c['ai_share'] > 0.5])}")
print(f"  • Clusters where AI > 40%: {len(ai_specific_clusters)}")
print(f"  • Maximum AI share in any cluster: {ai_in_tech_df['ai_share'].max()*100:.1f}%")

print(f"\nInterpretation:")
print(f"  AI concerns are DISTRIBUTED across the same concern clusters as other")
print(f"  technologies. There are no 'AI-only' concerns — the worries people express")
print(f"  about AI (privacy, control, transparency, governance) are the same worries")
print(f"  they express about nuclear, genetics, nanotechnology, etc.")

print(f"\nTech-specific clusters with most AI content:")
print(ai_in_tech_df.head(15)[['label', 'ai_count', 'ai_share', 'size']].to_string(index=False))

# Visualise: AI share across all tech-specific clusters
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ai_in_tech_df['ai_share'], bins=20, color='steelblue', edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', label='50% threshold')
ax.set_xlabel('AI Share of Cluster')
ax.set_ylabel('Number of Clusters')
ax.set_title('Distribution of AI Share Across Technology (metadata)-Specific Clusters\n(No cluster is dominated by AI)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "ai_share_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

ai_in_tech_df.to_csv(OUTPUT_FOLDER / "ai_in_tech_specific_clusters.csv", index=False)

show_complete("AI-specific concerns analysis complete")

In [ ]:
# pretty_label (concerns) — imported from dialogue_utils


## Stage 6 — Temporal trends and stability (concerns)

These cells track how concern themes change over time:

- **AI concern trajectory** — for each cluster, what fraction of AI dialogues per year
  mention it? This shows whether certain concerns become more or less prominent as the
  AI dialogue landscape matures.
- **Stability analysis** — which clusters persist across different values of k
  (number of clusters)? Stable clusters are more likely to reflect genuine themes
  rather than artefacts of the chosen k.
- **Salience** — within a given year, which concern clusters dominate?

In [ ]:
# normalized_entropy, hhi, topk_share, parse_year (concerns) — imported from dialogue_utils


In [ ]:
# @title Embed extracted concerns

show_status("Embedding extracted concerns...")

concern_embeddings_file = CHECKPOINT_FOLDER / "concern_embeddings.npy"

if concern_embeddings_file.exists():
    print("Found cached concern embeddings...")
    concern_embeddings = np.load(concern_embeddings_file)

else:
    concern_texts = concerns_df["concern"].astype(str).tolist()

    concern_embeddings = get_embeddings(
        concern_texts,
        client=client,
        model=EMBEDDING_MODEL
    )

    concern_embeddings = np.array(concern_embeddings)

    np.save(concern_embeddings_file, concern_embeddings)

show_complete(f"Embedded {len(concern_embeddings)} concerns")

In [ ]:
# @title Build AI concern trajectory over time

from sklearn.decomposition import PCA
import numpy as np
import pandas as pd

show_status("Building AI concern trajectory...")

# Check required inputs
required = ["concerns_df", "concern_embeddings"]

missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"Missing required variables: {missing}")

# Make sure embeddings align with concerns_df rows
embeddings = np.array(concern_embeddings)

if len(embeddings) != len(concerns_df):
    raise ValueError(
        f"concern_embeddings has {len(embeddings)} rows, "
        f"but concerns_df has {len(concerns_df)} rows. "
        "These must align before building a trajectory."
    )

# Filter to AI concerns with usable years
ai_mask = (
    concerns_df["technology"].astype(str).str.upper().eq("AI")
    & concerns_df["year"].notna()
)

ai_df = concerns_df.loc[ai_mask].copy()
ai_embeddings = embeddings[ai_mask.to_numpy()]

if len(ai_df) == 0:
    raise ValueError("No AI concerns with valid years found.")

# Compute yearly centroid in concern-embedding space
year_rows = []

for year in sorted(ai_df["year"].unique()):
    year_mask = ai_df["year"].eq(year).to_numpy()
    year_embeddings = ai_embeddings[year_mask]

    if len(year_embeddings) == 0:
        continue

    centroid = year_embeddings.mean(axis=0)

    year_rows.append({
        "year": int(year),
        "n_concerns": int(len(year_embeddings)),
        "centroid": centroid
    })

if len(year_rows) < 2:
    raise ValueError("Need at least two AI years to build a trajectory.")

centroids = np.vstack([row["centroid"] for row in year_rows])

# Project yearly centroids into 2D
pca = PCA(n_components=2)
coords = pca.fit_transform(centroids)

traj = pd.DataFrame({
    "year": [row["year"] for row in year_rows],
    "n_concerns": [row["n_concerns"] for row in year_rows],
    "pc1": coords[:, 0],
    "pc2": coords[:, 1],
})

traj.to_csv(OUTPUT_FOLDER / "ai_concern_trajectory.csv", index=False)

show_complete(f"Built AI concern trajectory for {len(traj)} years")

traj

In [ ]:
# @title Visualise AI concern trajectory over time

import matplotlib.pyplot as plt
import numpy as np

# Use traj DataFrame from the previous cell
# columns: year, pc1, pc2

traj2 = traj.sort_values("year").reset_index(drop=True)

# Center trajectory on first year
x0, y0 = traj2.loc[0, ["pc1", "pc2"]]
dx = traj2["pc1"] - x0
dy = traj2["pc2"] - y0

years = traj2["year"].to_numpy()

plt.figure(figsize=(6, 6))
sc = plt.scatter(dx, dy, c=years, cmap="viridis", s=80)
plt.plot(dx, dy, alpha=0.6)

plt.axhline(0, color="grey", linewidth=1, alpha=0.5)
plt.axvline(0, color="grey", linewidth=1, alpha=0.5)

plt.xlabel("Displacement in concern space (PC1)")
plt.ylabel("Displacement in concern space (PC2)")
plt.title("AI concern profile over time\n(displacement from initial position)")

cbar = plt.colorbar(sc)
cbar.set_label("Year")

plt.tight_layout()
plt.show()


In [ ]:
# @title Quantify stability of AI concern profiles
# Plot distance from initial position and from long-run mean

import numpy as np
import matplotlib.pyplot as plt

# traj2 already exists from previous cell
# columns: year, pc1, pc2

# Distance from first year
x0, y0 = traj2.loc[0, ["pc1", "pc2"]]
traj2["dist_from_start"] = np.sqrt(
    (traj2["pc1"] - x0)**2 + (traj2["pc2"] - y0)**2
)

# Distance from mean position
xm, ym = traj2[["pc1", "pc2"]].mean()
traj2["dist_from_mean"] = np.sqrt(
    (traj2["pc1"] - xm)**2 + (traj2["pc2"] - ym)**2
)

plt.figure(figsize=(9, 4.5))
plt.plot(traj2["year"], traj2["dist_from_start"], marker="o", label="Distance from initial year")
plt.plot(traj2["year"], traj2["dist_from_mean"], marker="o", label="Distance from long-run mean")

plt.xlabel("Year")
plt.ylabel("Distance in concern space")
plt.title("Stability of AI concern profile over time\n(smaller values = more stable)")
plt.legend()
plt.tight_layout()
plt.show()

display(traj2[["year", "dist_from_start", "dist_from_mean"]])


In [ ]:
# @title Build AI concern salience by year

show_status("Building AI concern salience by year...")

# Find the cluster column
if "cluster" in concerns_df.columns:
    cluster_col = "cluster"
elif "cluster_id" in concerns_df.columns:
    cluster_col = "cluster_id"
elif "concern_cluster" in concerns_df.columns:
    cluster_col = "concern_cluster"
else:
    raise NameError(
        "Could not find a cluster column in concerns_df. "
        "Expected one of: cluster, cluster_id, concern_cluster."
    )

# Filter to AI concerns with valid year and cluster assignment
ai_concerns_for_salience = concerns_df[
    concerns_df["technology"].astype(str).str.upper().eq("AI")
    & concerns_df["year"].notna()
    & concerns_df[cluster_col].notna()
].copy()

if len(ai_concerns_for_salience) == 0:
    raise ValueError("No AI concerns with year and cluster assignment found.")

ai_concerns_for_salience["year"] = ai_concerns_for_salience["year"].astype(int)
ai_concerns_for_salience[cluster_col] = ai_concerns_for_salience[cluster_col].astype(int)

# Raw salience: number of AI concern phrases per year per cluster
ai_year = pd.crosstab(
    ai_concerns_for_salience["year"],
    ai_concerns_for_salience[cluster_col]
).sort_index()

# Use readable cluster labels as columns where available
if "cluster_labels_list" in globals():
    ai_year = ai_year.rename(
        columns={
            c: cluster_labels_list[c] if c < len(cluster_labels_list) else f"Cluster {c}"
            for c in ai_year.columns
        }
    )

ai_year.to_csv(OUTPUT_FOLDER / "ai_concern_salience_by_year.csv")

show_complete(
    f"Built AI salience table: {ai_year.shape[0]} years × {ai_year.shape[1]} clusters"
)

ai_year

In [ ]:
# @title Analyse AI concern salience trajectories
# Shows which concern clusters rise or fall most within AI discourse over time.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year and years already exist from previous cells
# ai_year: years × clusters (raw salience)

# Normalize within each year so values sum to 1 (relative attention)
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# Compute linear trend (slope) for each cluster
# Using simple OLS slope over time
t = np.arange(len(ai_rel))  # time index (no assumption of equal year gaps)
slopes = {}
for c in ai_rel.columns:
    y = ai_rel[c].to_numpy()
    if np.all(y == 0):
        slopes[c] = np.nan
    else:
        # slope of y ~ t
        slopes[c] = np.polyfit(t, y, 1)[0]

trend = pd.Series(slopes).dropna().sort_values(ascending=False)

# Select top rising and declining clusters
N = 8
top_up = trend.head(N)
top_down = trend.tail(N)

# Plot trajectories
plt.figure(figsize=(11, 5))

# Rising
plt.subplot(1, 2, 1)
for c in top_up.index:
    plt.plot(ai_rel.index, ai_rel[c], marker="o", label=c)
plt.title("AI concerns increasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

# Declining
plt.subplot(1, 2, 2)
for c in top_down.index:
    plt.plot(ai_rel.index, ai_rel[c], marker="o", label=c)
plt.title("AI concerns decreasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

plt.tight_layout()
plt.show()

display(
    pd.DataFrame({
        "slope": trend
    }).assign(direction=lambda d: np.where(d["slope"] > 0, "rising", "declining"))
)


In [ ]:
# @title Visualise AI concern salience over time
# Rows = concern clusters
# Columns = years
# Values = relative salience within AI discourse (row-normalised per year)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year already exists: years × clusters (raw salience)
# years is a sorted array of years

# 1) Normalise within each year (relative attention)
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# 2) Select clusters to display
# Option A: top N by overall AI salience
N = 20
top_clusters = ai_rel.sum(axis=0).sort_values(ascending=False).head(N).index

heat = ai_rel[top_clusters]

# 3) Order clusters by when they peak (helps visual interpretation)
peak_year_idx = heat.values.argmax(axis=0)
order = np.argsort(peak_year_idx)
heat = heat.iloc[:, order]

# Transpose so clusters are rows
heat_T = heat.T

# 4) Plot heat map
plt.figure(figsize=(12, 7))
im = plt.imshow(
    heat_T,
    aspect="auto",
    cmap="viridis"
)

plt.colorbar(im, label="Share of AI attention (within year)")

plt.yticks(
    ticks=np.arange(len(heat_T.index)),
    labels=heat_T.index
)
plt.xticks(
    ticks=np.arange(len(heat_T.columns)),
    labels=heat_T.columns,
    rotation=45
)

plt.xlabel("Year")
plt.ylabel("Concern cluster")
plt.title("AI public concerns over time\n(relative salience within AI discourse)")

plt.tight_layout()
plt.show()


## Stage 7 — Core visualisations (concerns)

Summary visualisations of the concern landscape:
- **Stable core** — clusters with high prevalence and high technology entropy; these are
  the most robust cross-cutting concerns.
- **Embedding space** — 2D UMAP projection of all concern phrases, coloured by cluster.
  Useful for spotting cluster separation and potential merges.

In [ ]:
import os
print("CWD:", os.getcwd())
print("ARTIFACT_DIR exists:", os.path.exists("analysis_artifacts"))


In [ ]:
import os

ARTIFACT_DIR = "analysis_artifacts"
expected = "public_dialogue_analysis_v9.xlsx"

matches = []
for root, _, files in os.walk(ARTIFACT_DIR):
    if expected in files:
        matches.append(os.path.join(root, expected))

print("Matches:", matches)


In [ ]:
# @title Visualise the stable core of public concerns

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ---- prerequisites ----
if "cluster_entropy" not in globals():
    raise NameError("cluster_entropy not found. Run the cluster entropy section first.")
if "concerns_df" not in globals():
    raise NameError("concerns_df not found. Run concern extraction first.")

TECH_COL = "technology_meta"
ENTROPY_THRESHOLD = 0.5  # canonical cross-cutting threshold

# ---- build cluster-level dataframe ----
# cluster size = number of concern phrases
cluster_size = concerns_df["cluster_id"].value_counts()

df = (
    pd.DataFrame({
        "cluster_id": list(cluster_entropy.keys()),
        "entropy": [cluster_entropy[c] for c in cluster_entropy.keys()],
    })
    .set_index("cluster_id")
)

df["size"] = cluster_size
df["size"] = df["size"].fillna(0)
df = df[df["size"] > 0]

# cluster labels (if available)
if "CLUSTER_LABELS" in globals():
    df["label"] = df.index.map(
        lambda c: CLUSTER_LABELS.get(c, CLUSTER_LABELS.get(str(c), f"Cluster {c}"))
    )
else:
    df["label"] = df.index.map(lambda c: f"Cluster {c}")

entropy = df["entropy"].to_numpy(dtype=float)
size = df["size"].to_numpy(dtype=float)
labels = df["label"].to_numpy(dtype=str)

# ---- define cross-cutting + stable core ----
cross_cutting = entropy >= ENTROPY_THRESHOLD
size_thresh = np.quantile(size, 0.75)  # top 25% by frequency
stable_core = cross_cutting & (size >= size_thresh)

# clusters to annotate (most "core")
score = entropy * np.log1p(size)
annotate_idx = np.argsort(score)[::-1][:12]

# ---- plot ----
plt.figure(figsize=(10, 7))
ax = plt.gca()

# shaded stable-core region
core_region = Rectangle(
    (ENTROPY_THRESHOLD, size_thresh),
    1.0 - ENTROPY_THRESHOLD,
    size.max() * 1.05 - size_thresh,
    alpha=0.12,
    zorder=0
)
ax.add_patch(core_region)

ax.text(
    ENTROPY_THRESHOLD + 0.01,
    size.max() * 1.02,
    "Stable core\n(cross-cutting + high frequency)",
    fontsize=10,
    va="top"
)

plt.scatter(entropy[~cross_cutting], size[~cross_cutting], alpha=0.6, s=35, label="More tech-specific clusters")
plt.scatter(entropy[cross_cutting], size[cross_cutting], alpha=0.8, s=45, label="More cross-cutting clusters")
plt.scatter(entropy[stable_core], size[stable_core], s=90, label="Stable core")

for i in annotate_idx:
    plt.annotate(labels[i], (entropy[i], size[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

plt.axvline(ENTROPY_THRESHOLD, linestyle="--", linewidth=1)
plt.axhline(size_thresh, linestyle="--", linewidth=1)

plt.xlabel("Technology-distribution entropy (higher = more cross-cutting)")
plt.ylabel("Cluster frequency / size (higher = more recurrent)")
plt.title("Figure: The Stable Core of Public Technology Concerns")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# @title Visualise the concern embedding space
# Clusters projected into 2D using PCA over technology-salience vectors.
# Point size ~ cluster frequency proxy; color ~ framing lens (if available).

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ARTIFACT_DIR = globals().get("ARTIFACT_DIR", "analysis_artifacts")
EXCEL_PATH = globals().get("EXCEL_PATH", os.path.join(ARTIFACT_DIR, "public_dialogue_analysis_v9.xlsx"))

# 1) Load tech x cluster salience (in-memory)
if "salience_df" not in globals():
    raise NameError("salience_df not found. Run Section 6.1a to compute salience_df first.")

tech_by_cluster = salience_df.copy()
tech_by_cluster = tech_by_cluster.apply(pd.to_numeric, errors="coerce").fillna(0)


# Convert to clusters x technologies
cluster_features = tech_by_cluster.T  # rows=clusters, cols=technologies
clusters = cluster_features.index.astype(str).to_numpy()
X = cluster_features.to_numpy(dtype=float)

print("Cluster feature matrix:", X.shape, "(clusters x technologies)")

# 2) Size proxy (use total salience mass across technologies)
size_vals = cluster_features.sum(axis=1).to_numpy()
size_scaled = 30 + 200 * (np.sqrt(size_vals) / (np.sqrt(size_vals).max() + 1e-9))

# 3) Load framing lens mapping if available (optional)
lens_map = None
json_path = os.path.join(ARTIFACT_DIR, "framing_lens_mappings.json")
if os.path.exists(json_path):
    with open(json_path, "r") as f:
        lens_map = json.load(f)
    # Normalize to {cluster_label: lens}
    if lens_map and all(isinstance(v, list) for v in lens_map.values()):
        inv = {}
        for lens, clist in lens_map.items():
            for c in clist:
                inv[str(c)] = str(lens)
        lens_map = inv
    else:
        lens_map = {str(k): str(v) for k, v in lens_map.items()}

if lens_map is None:
    lens_labels = np.array(["(no lens)"] * len(clusters), dtype=object)
else:
    lens_labels = np.array([lens_map.get(c, "(unmapped)") for c in clusters], dtype=object)

# Encode lenses to integer codes
unique_lenses = pd.unique(lens_labels)
lens_to_code = {lens: i for i, lens in enumerate(unique_lenses)}
codes = np.array([lens_to_code[l] for l in lens_labels], dtype=int)

# 4) PCA projection (standardize across technologies)
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-9
Xz = (X - X_mean) / X_std

from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(Xz)
x, y = coords[:, 0], coords[:, 1]
expl = pca.explained_variance_ratio_

# 5) Plot
plt.figure(figsize=(10, 7))
ax = plt.gca()

# Use a categorical colormap; map integer codes -> colors
cmap = plt.get_cmap("tab20", len(unique_lenses))  # tab20 gives distinct-ish categories
sc = ax.scatter(x, y, s=size_scaled, c=codes, cmap=cmap, alpha=0.85)

ax.set_xlabel(f"Concern space (PC1; {expl[0]:.0%} var)")
ax.set_ylabel(f"Concern space (PC2; {expl[1]:.0%} var)")
ax.set_title("Figure: Concern Space of Public Technology Concerns\n(Clusters projected by technology-salience profile; size ~ frequency proxy)")

# Annotate top-N largest clusters (by size proxy)
topN = 12
top_idx = np.argsort(size_vals)[::-1][:topN]
for i in top_idx:
    ax.annotate(clusters[i], (x[i], y[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

# Legend: show up to 12 most frequent lenses (or all if <= 12)
lens_counts = pd.Series(lens_labels).value_counts()
top_lenses = lens_counts.index[:12].tolist() if len(lens_counts) > 12 else lens_counts.index.tolist()

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markersize=8,
           markerfacecolor=cmap(lens_to_code[lens]), markeredgecolor='none', label=lens)
    for lens in top_lenses
]
ax.legend(handles=legend_handles, title="Framing lens (top shown)", loc="best", frameon=True)

plt.tight_layout()
out_path = "figure_concern_space_pca.png"
plt.savefig(out_path, dpi=200)
plt.show()

print("Saved figure to:", out_path)


# Part II: Benefit analysis

This part mirrors the concern pipeline (Part I) for *benefits* — positive outcomes and
hopes that participants raised. The same extraction prompt, embedding model, clustering
approach, and analysis steps are used. Reading Part I's stage descriptions applies here too.

Key differences:
- A separate benefit extraction prompt with its own anti-artefact constraints
- `N_BENEFIT_CLUSTERS` controls the number of clusters (may differ from concern clusters)
- Outputs are prefixed `benefit_*` to avoid collision with concern outputs

## Stage 2B — LLM phrase extraction (benefits)

Each paragraph is sent to the LLM benefit extraction prompt, which asks for self-contained
phrases describing positive outcomes, hopes, or perceived advantages. The same caching,
tech-word filtering, and diagnostic outputs apply as in Stage 2.

In [ ]:
# @title Extract decontextualised benefit phrases

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import json
import pandas as pd

show_status("Extracting decontextualised benefit phrases...")

# BENEFIT_PROMPT is defined in dialogue_utils (includes anti-artefact rules)
# extract_benefits_from_paragraph replaced by extract_phrases(row, kind='benefit', ...) from dialogue_utils

# Cache file (benefits)
benefits_cache_file = CHECKPOINT_FOLDER / "extracted_benefits.json"

# Load cache if valid
if benefits_cache_file.exists():
    print("Found cached benefit extractions...")
    with open(benefits_cache_file) as f:
        cached_benefits = json.load(f)

    cached_ids = set(cached_benefits.keys())
    current_ids = set(chunks_df["chunk_id"].tolist())

    if cached_ids == current_ids:
        print(f"Using cached extractions for {len(cached_benefits)} paragraphs")
        if validate_extraction_cache(cached_benefits, "benefit"):
            all_benefits = cached_benefits
        else:
            print("Cache validation failed — will re-extract benefits")
            all_benefits = None
    else:
        print("Cache mismatch - will re-extract benefits")
        all_benefits = None
else:
    all_benefits = None

if all_benefits is None:
    all_benefits = {}

    # Metadata lookup
    chunk_metadata = chunks_df.set_index("chunk_id")[["technology", "year", "source_file"]].to_dict("index")

    MAX_WORKERS = 10
    rows = list(chunks_df.iterrows())

    benefit_results: list = []  # collect ExtractionResult objects for diagnostics

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extract_phrases, row, 'benefit', client, None, LLM_MODEL): row[1]["chunk_id"] for row in rows}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting benefits"):
            _result = future.result()
            benefit_results.append(_result)
            chunk_id, benefits = _result.chunk_id, _result.retained_phrases
            meta = chunk_metadata[chunk_id]
            all_benefits[chunk_id] = {
                "benefits": benefits,
                "technology": meta["technology"],
                "year": int(meta["year"]) if pd.notna(meta["year"]) else None,
                "source_file": meta["source_file"]
            }

    with open(benefits_cache_file, "w") as f:
        json.dump(all_benefits, f, indent=2)

    # Write CIP-0001 diagnostic files
    write_extraction_diagnostics(benefit_results, kind='benefit', output_folder=OUTPUT_FOLDER)

    show_complete(f"Extracted benefits from {len(all_benefits)} paragraphs")

# Flatten to individual benefit rows
benefit_rows = []
for chunk_id, data in all_benefits.items():
    for benefit in data["benefits"]:
        benefit_rows.append({
            "chunk_id": chunk_id,
            "benefit": benefit,
            "technology": data["technology"],
            "year": data["year"],
            "source_file": data["source_file"]
        })

benefits_df = pd.DataFrame(benefit_rows)
benefits_df["benefit_id"] = [f"benefit_{i}" for i in range(len(benefits_df))]

print(f"\nExtraction Summary:")
print(f"  Paragraphs processed: {len(all_benefits)}")
print(f"  Total benefit phrases: {len(benefits_df)}")
print(f"  Avg benefits per paragraph: {len(benefits_df) / max(1, len(all_benefits)):.2f}")
print(f"  Paragraphs with no benefits: {sum(1 for d in all_benefits.values() if len(d['benefits']) == 0)}")

benefits_df.to_csv(OUTPUT_FOLDER / "extracted_benefits.csv", index=False)        if not validate_extraction_cache(cached_benefits, "benefit"):
            print("Cache validation failed — will re-extract")
            all_benefits = None
        else:
            all_benefits = cached_benefits


In [ ]:
# Make sure benefits_df knows the technology of each paragraph (safe merge)

# Prefer chunks_df's 'technology' if 'technology_meta' doesn't exist there
tech_col = "technology_meta" if "technology_meta" in chunks_df.columns else "technology"

benefits_df = benefits_df.merge(
    chunks_df[["chunk_id", tech_col]].rename(columns={tech_col: "technology_meta"}),
    on="chunk_id",
    how="left"
)

# If benefits_df already has 'technology' and it's missing where technology_meta exists, fill it
if "technology" in benefits_df.columns:
    benefits_df["technology"] = benefits_df["technology"].fillna(benefits_df["technology_meta"])
else:
    benefits_df["technology"] = benefits_df["technology_meta"]

In [ ]:
# @title Inspect sample benefit extractions

import pandas as pd

print("Sample extracted benefits by technology:")
print("(Checking that technology-specific language has been removed)\n")

# Identify the text column in benefits_df
candidate_cols = ["benefit", "benefit_phrase", "phrase", "extracted_phrase", "text"]
phrase_col = next((c for c in candidate_cols if c in benefits_df.columns), None)

if phrase_col is None:
    raise KeyError(
        f"Couldn't find a benefit text column in benefits_df. "
        f"Looked for {candidate_cols}. Available columns: {list(benefits_df.columns)}"
    )

# Identify technology column (you appear to have technology_meta)
tech_col = "technology_meta" if "technology_meta" in benefits_df.columns else (
    "technology" if "technology" in benefits_df.columns else None
)

if tech_col is None:
    raise KeyError(
        f"Couldn't find a technology column in benefits_df. "
        f"Available columns: {list(benefits_df.columns)}"
    )

for tech in benefits_df[tech_col].dropna().unique()[:6]:
    tech_benefits = benefits_df[benefits_df[tech_col] == tech][phrase_col].head(8).tolist()
    print(f"\n{tech}:")
    for b in tech_benefits:
        print(f"  • {b}")

In [ ]:
# @title Vocabulary frequency diagnostic (benefits) — CIP-0004
# Flags meta-vocabulary terms that may be prompt artefacts.

vocab_benefit_df = vocabulary_frequency_diagnostic(
    phrases=benefits_df["benefit"].tolist(),
    kind="benefit",
    output_folder=OUTPUT_FOLDER,
)

_flagged = vocab_benefit_df[vocab_benefit_df["is_meta_vocab"]]
if not _flagged.empty:
    flagged_str = ", ".join(
        f'"{r["term"]}": {r["pct_of_phrases"]}%' for _, r in _flagged.iterrows()
    )
    show_warning(f"Meta-vocabulary detected in benefit phrases: {flagged_str}")
else:
    show_complete("No meta-vocabulary terms in top-100 benefit terms.")

vocab_benefit_df.head(20)


## Stage 3B — Semantic embeddings and clustering (benefits)

Benefit phrases are embedded, normalised, and clustered using the same approach as concerns.
Technology entropy is computed to classify clusters as cross-cutting vs. technology-specific.

In [ ]:
# @title Generate semantic embeddings for benefit phrases

# get_embeddings_batch imported from dialogue_utils

embeddings_file = CHECKPOINT_FOLDER / "benefit_embeddings.npy"
benefit_ids_file = CHECKPOINT_FOLDER / "benefit_ids.json"

if embeddings_file.exists() and benefit_ids_file.exists():
    print("Found cached embeddings...")
    embeddings = np.load(embeddings_file)
    with open(benefit_ids_file) as f:
        cached_benefit_ids = json.load(f)

    if cached_benefit_ids == benefits_df["benefit_id"].tolist():
        print(f"Using cached embeddings: {embeddings.shape}")
    else:
        print("Cache mismatch - regenerating")
        embeddings = None
else:
    embeddings = None

if embeddings is None:
    show_status(f"Generating embeddings for {len(benefits_df)} benefit phrases...")

    texts = benefits_df["benefit"].tolist()
    all_embeddings = []

    for i in tqdm(range(0, len(texts), EMBEDDING_BATCH_SIZE), desc="Embedding"):
        batch = texts[i:i+EMBEDDING_BATCH_SIZE]
        try:
            batch_embeddings = get_embeddings_batch(batch, client, model=EMBEDDING_MODEL)
            all_embeddings.append(batch_embeddings)
        except Exception as e:
            print(f"Error on batch {i}: {e}")
        time.sleep(0.1)

    embeddings = np.vstack(all_embeddings) if all_embeddings else np.empty((0, 0))
    np.save(embeddings_file, embeddings)
    with open(benefit_ids_file, "w") as f:
        json.dump(benefits_df["benefit_id"].tolist(), f)

    show_complete(f"Generated embeddings: {embeddings.shape}")

if embeddings.size:
    print(f"Embedding dimensions: {embeddings.shape[1]}")
else:
    print("No embeddings generated (benefits_df empty).")

In [ ]:
# @title Cluster benefit phrase embeddings

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)

show_status(f"Clustering into {N_BENEFIT_CLUSTERS} benefit groups...")

benefit_embeddings_normalized = normalize(embeddings)

kmeans = KMeans(
    n_clusters=N_BENEFIT_CLUSTERS,
    random_state=RANDOM_SEED,
    n_init=10,
    max_iter=300
)
cluster_labels = kmeans.fit_predict(embeddings_normalized)

centroids = kmeans.cluster_centers_
benefit_centroids_normalized = normalize(centroids)

# Add cluster assignment to benefits dataframe
benefits_df["cluster_id"] = cluster_labels

# SOFT MEMBERSHIP: cosine similarity to each centroid
show_status("Computing soft membership scores...")
soft_membership = cosine_similarity(embeddings_normalized, centroids_normalized)

print(f"\nClustering Results:")
print(f"  Clusters: {N_BENEFIT_CLUSTERS}")
print(f"  Benefit phrases: {len(benefits_df)}")
print(f"  Soft membership matrix: {soft_membership.shape}")

cluster_sizes = np.bincount(cluster_labels)
print(f"  Cluster sizes: min={cluster_sizes.min()}, max={cluster_sizes.max()}, median={np.median(cluster_sizes):.0f}")

np.save(CHECKPOINT_FOLDER / "benefit_soft_membership.npy", soft_membership)
np.save(CHECKPOINT_FOLDER / "benefit_cluster_centroids.npy", centroids_normalized)

show_complete("Benefit clustering complete")

In [ ]:
# @title Characterise benefit clusters by technology distribution

from scipy.stats import entropy as scipy_entropy

# @title Characterise clusters by technology distribution

show_status("Analysing cluster composition...")

# For each cluster, compute entropy of technology distribution
cluster_entropy = {}
cluster_tech_dist = {}

technologies = benefits_df['technology_meta'].unique()
n_techs = len(technologies)

for cluster_id in range(N_BENEFIT_CLUSTERS):
    cluster_mask = benefits_df['cluster_id'] == cluster_id
    cluster_data = benefits_df[cluster_mask]

    if len(cluster_data) == 0:
        cluster_entropy[cluster_id] = 0
        cluster_tech_dist[cluster_id] = {}
        continue

    # Technology distribution
    tech_counts = cluster_data['technology_meta'].value_counts()
    tech_probs = tech_counts / tech_counts.sum()

    # Entropy (higher = more cross-cutting)
    cluster_entropy[cluster_id] = scipy_entropy(tech_probs)
    cluster_tech_dist[cluster_id] = tech_counts.to_dict()

# Normalize entropy to 0-1 scale (max entropy = log(n_techs))
max_entropy = np.log(n_techs)
normalized_entropy_benefits = {k: v / max_entropy for k, v in cluster_entropy.items()}

# Classify clusters
CROSS_CUTTING_THRESHOLD = CROSSCUTTING_ENTROPY_THRESHOLD  # from dialogue_utils

cross_cutting_clusters_benefits = [k for k, v in normalized_entropy_benefits.items() if v >= CROSS_CUTTING_THRESHOLD]
tech_specific_clusters_benefits = [k for k, v in normalized_entropy_benefits.items() if v < CROSS_CUTTING_THRESHOLD]

print(f"\nCluster Classification:")
print(f"  Cross-cutting clusters (entropy >= {CROSS_CUTTING_THRESHOLD}): {len(cross_cutting_clusters_benefits)}")
print(f"  Technology-specific clusters: {len(tech_specific_clusters_benefits)}")
print(f"  Ratio: {len(cross_cutting_clusters_benefits)/N_BENEFIT_CLUSTERS*100:.1f}% cross-cutting")

# Visualise entropy distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of entropy
axes[0].hist(list(normalized_entropy_benefits.values()), bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({CROSS_CUTTING_THRESHOLD})')
axes[0].set_xlabel('Normalized Entropy')
axes[0].set_ylabel('Number of Clusters')
axes[0].set_title('Cluster Entropy Distribution\n(Higher = More Cross-Cutting)')
axes[0].legend()

# Entropy vs cluster size
sizes = [int((benefits_df["cluster_id"] == i).sum()) for i in range(N_BENEFIT_CLUSTERS)]
entropies = [normalized_entropy_benefits[i] for i in range(N_BENEFIT_CLUSTERS)]
colors = ['green' if e >= CROSS_CUTTING_THRESHOLD else 'orange' for e in entropies]
axes[1].scatter(sizes, entropies, c=colors, alpha=0.6)
axes[1].axhline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--')
axes[1].set_xlabel('Cluster Size')
axes[1].set_ylabel('Normalized Entropy')
axes[1].set_title('Cross-Cutting (green) vs Technology-Specific (orange)')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "cluster_entropy_analysis.png", dpi=150)
plt.show()

show_complete("Cluster composition analysis complete")

In [ ]:
# --- Define cross-cutting helpers for Benefits (Section 6) ---

CROSS_CUTTING_ENTROPY_THRESHOLD = globals().get("CROSS_CUTTING_THRESHOLD", 0.5)

if "cross_cutting_clusters_benefits" not in globals():
    raise NameError("cross_cutting_clusters_benefits not found. Run the benefit entropy/classification cell first.")

cross_cutting_cluster_ids_benefits = set(cross_cutting_clusters_benefits)

label_map = {}
if "benefit_summary_df" in globals() and {"cluster_id", "label"}.issubset(benefit_summary_df.columns):
    label_map = dict(zip(benefit_summary_df["cluster_id"], benefit_summary_df["label"]))

cross_cutting_labels_benefits = {
    cid: label_map.get(cid, f"Cluster {cid}") for cid in cross_cutting_cluster_ids_benefits
}

## Stage 4B — Cluster labelling (benefits)

Exemplar benefit phrases are sent to the LLM for labelling. As with concerns, technology
metadata is excluded from the prompt to avoid biasing labels.

In [ ]:
# @title Extract exemplar benefit phrases per cluster

show_status("Extracting exemplar benefits...")

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)
N_EXEMPLARS = 8
benefit_cluster_exemplars = {}

for cluster_id in range(N_BENEFIT_CLUSTERS):
    cluster_mask = benefits_df["cluster_id"] == cluster_id
    cluster_benefits = benefits_df[cluster_mask]
    cluster_embs = benefit_embeddings_normalized[cluster_mask]

    if len(cluster_benefits) == 0:
        continue

    centroid = benefit_centroids_normalized[cluster_id]
    similarities = cosine_similarity(cluster_embs, centroid.reshape(1, -1)).flatten()
    top_indices = np.argsort(similarities)[-N_EXEMPLARS:][::-1]

    exemplars = []
    for i in top_indices:
        row = cluster_benefits.iloc[i]
        exemplars.append({
            "benefit": row["benefit"],
            "technology": row["technology"],
            "year": int(row["year"]) if pd.notna(row["year"]) else None,
            "similarity": float(similarities[i])
        })

    tech_dist = cluster_benefits["technology"].value_counts().head(3).to_dict()

    benefit_cluster_exemplars[cluster_id] = {
        "size": len(cluster_benefits),
        "entropy": normalized_entropy_benefits[cluster_id],
        "is_cross_cutting": cluster_id in cross_cutting_clusters_benefits,
        "top_technologies": tech_dist,
        "exemplars": exemplars
    }

with open(OUTPUT_FOLDER / "benefit_cluster_exemplars.json", "w") as f:
    json.dump(benefit_cluster_exemplars, f, indent=2, default=str)

show_complete(f"Extracted exemplars for {len(benefit_cluster_exemplars)} clusters")

In [ ]:
# label_cluster (benefits) — duplicate removed; imported from dialogue_utils


## Stage 4C — Framing lenses (benefits)

Benefit clusters are mapped to high-level framing lenses (e.g. "efficiency", "safety",
"equity") using the LLM. This provides a higher-level vocabulary for comparing concern
and benefit framing structures.

In [ ]:
# @title Identify benefit framing lenses

show_status("Generating benefit framing lens suggestions...")

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)

# Prepare cluster summary for GPT
cluster_info = []
for cluster_id, data in benefit_cluster_exemplars.items():
    label = cluster_labels_dict.get(str(cluster_id), {}).get('label', f'Cluster {cluster_id}')
    description = cluster_labels_dict.get(str(cluster_id), {}).get('description', '')
    cluster_type = 'cross-cutting' if data['is_cross_cutting'] else 'tech-specific'
    cluster_info.append(f"{cluster_id}. {label} ({cluster_type}, n={data['size']}): {description}")

cluster_summary_text = "\n".join(cluster_info)

lens_prompt = f"""Analyze these {N_BENEFIT_CLUSTERS} benefit clusters from UK public dialogue reports.

Clusters:
{cluster_summary_text}

Group these clusters into 8-12 higher-level FRAMING LENSES that capture how publics frame their benefits.

For each lens provide:
1. Name (2-4 words)
2. Description (1 sentence)
3. List of cluster IDs that belong to this lens

A cluster can belong to multiple lenses if appropriate.
Ensure all clusters are assigned to at least one lens.

Return JSON:
{{"framing_lenses": [
  {{"name": "...", "description": "...", "suggested_clusters": [0, 1, 2...]}},
  ...
]}}"""

try:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "Expert in public engagement and discourse analysis. Return only valid JSON."},
            {"role": "user", "content": lens_prompt}
        ],
        max_completion_tokens=8000
    )

    content = response.choices[0].message.content
    if '```' in content:
        parts = content.split('```')
        for part in parts:
            if part.startswith('json'):
                content = part[4:].strip()
                break
            elif part.strip().startswith('{'):
                content = part.strip()
                break

    suggested_lenses = json.loads(content)

    print("Suggested Benefit Framing Lenses:\n")
    for lens in suggested_lenses['framing_lenses']:
        n_clusters = len(lens['suggested_clusters'])
        print(f"  {lens['name']}")
        print(f"    {lens['description']}")
        print(f"    Clusters: {lens['suggested_clusters'][:8]}...\n")

    show_complete(f"Generated {len(suggested_lenses['framing_lenses'])} benefit framing lenses")

except Exception as e:
    print(f"Error generating lenses: {e}")
    suggested_lenses = {'framing_lenses': []}

In [ ]:
# @title Map benefit clusters to framing lenses

FRAMING_LENS_MAPPINGS = {}
for lens in suggested_lenses['framing_lenses']:
    FRAMING_LENS_MAPPINGS[lens['name']] = {
        'description': lens['description'],
        'cluster_ids': lens['suggested_clusters']
    }

with open(OUTPUT_FOLDER / "benefit_framing_lens_mappings.json", 'w') as f:
    json.dump(FRAMING_LENS_MAPPINGS, f, indent=2)

# Map clusters to lenses
cluster_to_lens = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    for cluster_id in data['cluster_ids']:
        if cluster_id not in cluster_to_lens:
            cluster_to_lens[cluster_id] = []
        cluster_to_lens[cluster_id].append(lens_name)

print("Benefit Framing Lens Mappings:")
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    n_clusters = len(data['cluster_ids'])
    lens_mask = benefits_df['cluster_id'].isin(data['cluster_ids'])
    n_benefits = lens_mask.sum()
    print(f"  {lens_name}: {n_clusters} clusters, {n_benefits} benefits")

In [ ]:
# @title Compute benefit framing lens prevalence

show_status("Computing benefit framing lens prevalence...")

lens_prevalence = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_mask = benefits_df['cluster_id'].isin(data['cluster_ids'])
    lens_prevalence[lens_name] = lens_mask.sum()

lens_prev_df = pd.DataFrame([
    {'Framing Lens': k, 'Benefits': v}
    for k, v in lens_prevalence.items()
]).sort_values('Benefits', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(lens_prev_df['Framing Lens'], lens_prev_df['Benefits'], color='steelblue')
ax.set_xlabel('Number of Benefit Phrases')
ax.set_title('Benefit Framing Lens Prevalence Across All Documents')

for bar, val in zip(bars, lens_prev_df['Benefits']):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_framing_lens_prevalence.png", dpi=150, bbox_inches='tight')
plt.show()

show_complete("Benefit framing lens prevalence computed")

## Stage 5B — Cross-cutting analysis and AI distinctiveness (benefits)

The same cross-cutting and AI-fingerprint analysis is applied to benefit clusters.
This identifies which benefits participants expect across technologies generally,
and which are especially prominent in AI dialogues.

In [ ]:
# --- Section 6 prerequisites (canonical tech + cross-cutting clusters) ---

TECH_COL = "technology_meta"

# Ensure benefits_df has technology_meta
if TECH_COL not in benefits_df.columns:
    benefits_df = benefits_df.merge(
        chunks_df[["chunk_id", TECH_COL]],
        on="chunk_id",
        how="left"
    )

# Define cross-cutting clusters from entropy (single source of truth)
CROSS_CUTTING_ENTROPY_THRESHOLD = 0.5
cross_cutting_cluster_ids_benefits = {
    cid for cid, ent in cluster_entropy.items()
    if ent >= CROSS_CUTTING_ENTROPY_THRESHOLD
}


In [ ]:
# @title Shared benefits across technologies (document-weighted)

show_status("Computing shared benefit structure (all technologies)...")

TECH_COL = "technology_meta"

# Canonical list of technologies from metadata
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())

# Salience matrix: rows=technology, cols=cluster_id, values=proportion of benefits
salience_rows = {}
for tech in technologies:
    tech_mask = benefits_df[TECH_COL] == tech
    tech_total = int(tech_mask.sum())
    if tech_total == 0:
        continue
    counts = benefits_df.loc[tech_mask, "cluster_id"].value_counts()
    salience_rows[tech] = (counts / tech_total).to_dict()

salience_df = pd.DataFrame(salience_rows).T.fillna(0)

# Document-weighted global prevalence of clusters
global_cluster_prevalence = benefits_df["cluster_id"].value_counts(normalize=True)

cluster_meta_df = (
    pd.DataFrame(
        {
            "cluster_id": list(cluster_entropy.keys()),
            "tech_entropy": [cluster_entropy[c] for c in cluster_entropy.keys()],
            "global_prevalence": [global_cluster_prevalence.get(c, 0) for c in cluster_entropy.keys()],
        }
    )
    .set_index("cluster_id")
    .sort_values(["global_prevalence", "tech_entropy"], ascending=False)
)

# Shared benefits = high prevalence + high entropy
shared_benefits = cluster_meta_df.head(20)

print("Top shared benefit clusters (high prevalence + cross-technology spread):")
display(shared_benefits.head(12))

plt.figure(figsize=(10, 5))
shared_benefits.sort_values("global_prevalence")["global_prevalence"].plot(kind="barh", legend=False)
plt.title("Shared benefits across technologies (top 20 clusters by prevalence x spread)")
plt.xlabel("Share of all extracted benefit phrases")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_benefits_top20.png")
plt.show()

shared_benefits.reset_index().to_csv(OUTPUT_FOLDER / "shared_benefits_top20.csv", index=False)

show_complete("Shared benefits computed")

In [ ]:
# @title Shared benefit framings across technologies (document-weighted)

from scipy.stats import entropy as scipy_entropy

show_status("Computing shared benefit framing structure (all technologies)...")

TECH_COL = 'technology_meta'
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())

# Lens prevalence overall + entropy across technologies
lens_stats = []
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_clusters = set(data['cluster_ids'])
    lens_mask = benefits_df['cluster_id'].isin(lens_clusters)
    overall_prev = lens_mask.mean()

    tech_counts = benefits_df.loc[lens_mask, TECH_COL].value_counts()
    tech_probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum() > 0 else np.array([])
    lens_entropy = float(scipy_entropy(tech_probs)) if len(tech_probs) > 1 else 0.0

    lens_stats.append({
        'framing_lens': lens_name,
        'overall_prevalence': float(overall_prev),
        'tech_entropy': lens_entropy,
        'n_benefits_in_lens': int(lens_mask.sum())
    })

lens_meta_df = (pd.DataFrame(lens_stats)
                .sort_values(['overall_prevalence','tech_entropy'], ascending=False))

print("Shared benefit framings (high prevalence + cross-technology spread):")
display(lens_meta_df.head(12))

plt.figure(figsize=(7,5))
plt.scatter(lens_meta_df['tech_entropy'], lens_meta_df['overall_prevalence'])
for _, r in lens_meta_df.iterrows():
    plt.text(r['tech_entropy'], r['overall_prevalence'], r['framing_lens'], fontsize=8)
plt.xlabel("Entropy across technologies")
plt.ylabel("Share of all extracted benefits")
plt.title("Shared benefit framings across technologies")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_benefit_framings_scatter.png")
plt.show()

lens_meta_df.to_csv(OUTPUT_FOLDER / "shared_benefit_framings.csv", index=False)
show_complete("Shared benefit framings computed")

In [ ]:
# @title Compare AI and non-AI dialogues by framing lens

show_status("Computing AI vs non-AI framing profile (technology-weighted baseline)...")

TECH_COL = 'technology_meta'
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Lens salience per technology (within-technology proportions)
lens_salience_by_tech = {}
for tech in technologies:
    tech_mask = benefits_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    lens_salience_by_tech[tech] = {}
    for lens_name, data in FRAMING_LENS_MAPPINGS.items():
        lens_mask = benefits_df['cluster_id'].isin(data['cluster_ids'])
        lens_salience_by_tech[tech][lens_name] = float((tech_mask & lens_mask).sum() / tech_total)

lens_salience_df = pd.DataFrame(lens_salience_by_tech).T.fillna(0)

# Technology-weighted non-AI baseline (each non-AI technology counts equally)
non_ai_avg = lens_salience_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

# Plot radar: AI vs non-AI avg
if 'AI' in lens_salience_df.index and len(non_ai_avg) > 0:
    categories = list(non_ai_avg.index)
    ai_vals = lens_salience_df.loc['AI', categories].tolist()
    base_vals = non_ai_avg[categories].tolist()

    # Close the loop
    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI benefit framing profile vs non‑AI baseline (technology‑weighted)"
    )
    fig.write_html(OUTPUT_FOLDER / "benefit_ai_vs_nonai_framing_radar.html")
    fig.show()

# Save table
lens_salience_df.to_csv(OUTPUT_FOLDER / "benefit_lens_salience_by_technology_meta.csv")
non_ai_avg.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "benefit_non_ai_avg_framing_profile.csv")
show_complete("AI vs non-AI framing profile computed")

In [ ]:
# @title Compare AI and non-AI dialogues on cross-cutting concerns

show_status("Computing AI vs non-AI benefit profile on cross-cutting clusters...")

TECH_COL = 'technology_meta'
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Use cross_cutting_labels_benefits from SECTION 3 (derived from entropy threshold)
cross_cutting_cluster_ids_benefits = set(cross_cutting_labels_benefits.keys())

# Technology profiles over cross-cutting clusters
profiles = {}
for tech in technologies:
    tech_mask = benefits_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    counts = benefits_df.loc[tech_mask & benefits_df['cluster_id'].isin(cross_cutting_cluster_ids_benefits), 'cluster_id'].value_counts()
    # normalise over all concerns for that tech (so "attention share" to cross-cutting clusters)
    profiles[tech] = (counts / tech_total).to_dict()

profiles_df = pd.DataFrame(profiles).T.fillna(0)

non_ai_avg = profiles_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

# Save for later use
profiles_df.to_csv(OUTPUT_FOLDER / "cross_cutting_cluster_profiles_by_tech.csv")
non_ai_avg.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "cross_cutting_cluster_profile_non_ai_avg.csv")

show_complete("Cross-cutting concern profiles computed")


In [ ]:
# --- Build CLUSTER_LABELS dict for plots (cluster_id -> label) ---
if "benefit_summary_df" in globals() and {"cluster_id", "label"}.issubset(benefit_summary_df.columns):
    CLUSTER_LABELS = dict(zip(benefit_summary_df["cluster_id"], benefit_summary_df["label"]))
else:
    CLUSTER_LABELS = {}

In [ ]:
# @title Visualise AI benefit fingerprint (AI vs non-AI)

TECH_COL = 'technology_meta'

if 'AI' in profiles_df.index and len(non_ai_avg) > 0:
    show_status("Creating AI fingerprint radar (AI vs non-AI)...")

    diff = (profiles_df.loc['AI'] - non_ai_avg).sort_values()
    top_low = diff.head(5).index.tolist()
    top_high = diff.tail(7).index.tolist()
    selected = top_low + top_high

    # Human-friendly labels
    def pretty_cluster(cid):
        return CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    categories = [pretty_cluster(c) for c in selected]
    ai_vals = profiles_df.loc['AI', selected].tolist()
    base_vals = non_ai_avg[selected].tolist()

    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI benefit fingerprint vs non‑AI baseline (cross‑cutting clusters)"
    )
    fig.write_html(OUTPUT_FOLDER / "benefit_ai_vs_nonai_radar.html")
    fig.show()

show_complete("AI fingerprint radar created")

In [ ]:
# @title Identify distinctive AI benefits and framings

show_status("Computing distinctive benefits and framings for AI...")

TECH_COL = 'technology_meta'
non_ai_techs = [t for t in technologies if t != 'AI']

# Distinctive benefit clusters (all clusters, tech-weighted baseline)
all_cluster_profiles = salience_df  # rows=tech, cols=cluster_id

if 'AI' in all_cluster_profiles.index and len(non_ai_techs) > 0:
    non_ai_avg_all = all_cluster_profiles.loc[non_ai_techs].mean(axis=0)
    ai_vs_base = (all_cluster_profiles.loc['AI'] - non_ai_avg_all).sort_values()

    most_under = ai_vs_base.head(10)
    most_over = ai_vs_base.tail(10)

    def label(cid):
        return CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    distinctive_benefits = pd.DataFrame({
        'cluster_id': list(most_under.index) + list(most_over.index),
        'cluster_label': [label(c) for c in list(most_under.index) + list(most_over.index)],
        'ai_minus_nonai': list(most_under.values) + list(most_over.values),
        'ai_share': [float(all_cluster_profiles.loc['AI', c]) for c in list(most_under.index) + list(most_over.index)],
        'nonai_share': [float(non_ai_avg_all[c]) for c in list(most_under.index) + list(most_over.index)],
    }).sort_values('ai_minus_nonai')

    display(distinctive_benefits)
    distinctive_benefits.to_csv(OUTPUT_FOLDER / "ai_distinctive_benefits.csv", index=False)

# Distinctive framings (lens level)
if 'AI' in lens_salience_df.index and len(non_ai_techs) > 0:
    non_ai_avg_lens = lens_salience_df.loc[non_ai_techs].mean(axis=0)
    lens_diff = (lens_salience_df.loc['AI'] - non_ai_avg_lens).sort_values()

    df = pd.DataFrame({
        'framing_lens': lens_diff.index,
        'ai_minus_nonai': lens_diff.values,
        'ai_share': lens_salience_df.loc['AI', lens_diff.index].values,
        'nonai_share': non_ai_avg_lens[lens_diff.index].values
    }).sort_values('ai_minus_nonai')

    display(df.head(10))
    display(df.tail(10))
    df.to_csv(OUTPUT_FOLDER / "benefit_ai_distinctive_framings.csv", index=False)

show_complete("Distinctiveness outputs saved")

In [ ]:
# @title Analyse AI-specific benefits

show_status("Identifying AI-specific benefits...")

# Find clusters that are predominantly AI
ai_specific_clusters = []

for cluster_id in tech_specific_clusters_benefits:
    tech_dist = cluster_tech_dist.get(cluster_id, {})
    total = sum(tech_dist.values())
    if total == 0:
        continue

    ai_count = tech_dist.get('AI', 0)
    ai_share = ai_count / total

    if ai_share > 0.4:
        ai_specific_clusters.append({
            'cluster_id': cluster_id,
            'label': benefit_cluster_labels_list[cluster_id],
            'size': benefit_cluster_exemplars.get(cluster_id, {}).get('size', 0),
            'ai_share': ai_share,
            'ai_count': ai_count
        })

# Collect AI presence in all tech-specific clusters
ai_in_tech_specific = []
for cluster_id in tech_specific_clusters_benefits:
    tech_dist = cluster_tech_dist.get(cluster_id, {})
    ai_count = tech_dist.get('AI', 0)
    total = sum(tech_dist.values())
    ai_in_tech_specific.append({
        'cluster_id': cluster_id,
        'label': benefit_cluster_labels_list[cluster_id],
        'size': benefit_cluster_exemplars.get(cluster_id, {}).get('size', 0),
        'ai_count': ai_count,
        'ai_share': ai_count / total if total > 0 else 0
    })

ai_in_tech_df = pd.DataFrame(ai_in_tech_specific).sort_values('ai_count', ascending=False)

print("="*70)
print("KEY FINDING: AI Benefits Are Not Distinctive at the Cluster Level")
print("="*70)

print(f"\nOf {len(tech_specific_clusters_benefits)} technology-specific clusters:")
print(f"  • Clusters where AI > 50%: {len([c for c in ai_specific_clusters if c['ai_share'] > 0.5])}")
print(f"  • Clusters where AI > 40%: {len(ai_specific_clusters)}")
print(f"  • Maximum AI share in any cluster: {ai_in_tech_df['ai_share'].max()*100:.1f}%")

print(f"\nInterpretation:")
print(f"  AI benefits are DISTRIBUTED across the same benefit clusters as other")
print(f"  technologies. There are no 'AI-only' benefits — the gains people expect")
print(f"  from AI are the same gains they expect from nuclear, genetics, etc.")

print(f"\nTech-specific clusters with most AI content:")
print(ai_in_tech_df.head(15)[['label', 'ai_count', 'ai_share', 'size']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ai_in_tech_df['ai_share'], bins=20, color='steelblue', edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', label='50% threshold')
ax.set_xlabel('AI Share of Cluster')
ax.set_ylabel('Number of Clusters')
ax.set_title('Distribution of AI Share Across Technology-Specific Benefit Clusters')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_ai_share_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

ai_in_tech_df.to_csv(OUTPUT_FOLDER / "benefit_ai_in_tech_specific_clusters.csv", index=False)

show_complete("AI-specific benefits analysis complete")

In [ ]:
# pretty_label (benefits) — duplicate removed; imported from dialogue_utils


## Stage 6B — Temporal trends and stability (benefits)

Benefit cluster prevalence over time, stability across k values, and AI salience
trajectories — mirroring the concern temporal analysis in Stage 6.

In [ ]:
# normalized_entropy, hhi, topk_share, parse_year (benefits) — duplicate removed; imported from dialogue_utils


In [ ]:
# @title Visualise AI benefit trajectory over time

import matplotlib.pyplot as plt
import numpy as np

# Use traj DataFrame from the previous cell
# columns: year, pc1, pc2

traj2 = traj.sort_values("year").reset_index(drop=True)

# Center trajectory on first year
x0, y0 = traj2.loc[0, ["pc1", "pc2"]]
dx = traj2["pc1"] - x0
dy = traj2["pc2"] - y0

years = traj2["year"].to_numpy()

plt.figure(figsize=(6, 6))
sc = plt.scatter(dx, dy, c=years, cmap="viridis", s=80)
plt.plot(dx, dy, alpha=0.6)

plt.axhline(0, color="grey", linewidth=1, alpha=0.5)
plt.axvline(0, color="grey", linewidth=1, alpha=0.5)

plt.xlabel("Displacement in benefit space (PC1)")
plt.ylabel("Displacement in benefit space (PC2)")
plt.title("AI benefit profile over time\n(displacement from initial position)")

cbar = plt.colorbar(sc)
cbar.set_label("Year")

plt.tight_layout()
plt.show()

In [ ]:
# @title Quantify stability of AI benefit profiles
# Plot distance from initial position and from long-run mean

import numpy as np
import matplotlib.pyplot as plt

# traj2 already exists from previous cell
# columns: year, pc1, pc2

# Distance from first year
x0, y0 = traj2.loc[0, ["pc1", "pc2"]]
traj2["dist_from_start"] = np.sqrt(
    (traj2["pc1"] - x0)**2 + (traj2["pc2"] - y0)**2
)

# Distance from mean position
xm, ym = traj2[["pc1", "pc2"]].mean()
traj2["dist_from_mean"] = np.sqrt(
    (traj2["pc1"] - xm)**2 + (traj2["pc2"] - ym)**2
)

plt.figure(figsize=(9, 4.5))
plt.plot(traj2["year"], traj2["dist_from_start"], marker="o", label="Distance from initial year")
plt.plot(traj2["year"], traj2["dist_from_mean"], marker="o", label="Distance from long-run mean")

plt.xlabel("Year")
plt.ylabel("Distance in benefit space")
plt.title("Stability of AI benefit profile over time\n(smaller values = more stable)")
plt.legend()
plt.tight_layout()
plt.show()

display(traj2[["year", "dist_from_start", "dist_from_mean"]])

In [ ]:
# @title Analyse AI benefit salience trajectories
# Shows which benefit clusters rise or fall most within AI discourse over time.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year and years already exist from previous cells
# ai_year: years × clusters (raw salience)

# Normalize within each year so values sum to 1 (relative attention)
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# Compute linear trend (slope) for each cluster
# Using simple OLS slope over time
t = np.arange(len(ai_rel))  # time index (no assumption of equal year gaps)
slopes = {}
for c in ai_rel.columns:
    y = ai_rel[c].to_numpy()
    if np.all(y == 0):
        slopes[c] = np.nan
    else:
        # slope of y ~ t
        slopes[c] = np.polyfit(t, y, 1)[0]

trend = pd.Series(slopes).dropna().sort_values(ascending=False)

# Select top rising and declining clusters
N = 8
top_up = trend.head(N)
top_down = trend.tail(N)

# Plot trajectories
plt.figure(figsize=(11, 5))

# Rising
plt.subplot(1, 2, 1)
for c in top_up.index:
    plt.plot(ai_rel.index, ai_rel[c], marker="o", label=c)
plt.title("AI benefits increasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

# Declining
plt.subplot(1, 2, 2)
for c in top_down.index:
    plt.plot(ai_rel.index, ai_rel[c], marker="o", label=c)
plt.title("AI benefits decreasing in relative salience")
plt.xlabel("Year")
plt.ylabel("Share of AI attention")
plt.legend(fontsize=9)

plt.tight_layout()
plt.show()

display(
    pd.DataFrame({
        "slope": trend
    }).assign(direction=lambda d: np.where(d["slope"] > 0, "rising", "declining"))
)

In [ ]:
# @title Visualise AI benefit salience over time
# Rows = benefit clusters
# Columns = years
# Values = relative salience within AI discourse (row-normalised per year)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year already exists: years × clusters (raw salience)
# years is a sorted array of years

# 1) Normalise within each year (relative attention)
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# 2) Select clusters to display
# Option A: top N by overall AI salience
N = 20
top_clusters = ai_rel.sum(axis=0).sort_values(ascending=False).head(N).index

heat = ai_rel[top_clusters]

# 3) Order clusters by when they peak (helps visual interpretation)
peak_year_idx = heat.values.argmax(axis=0)
order = np.argsort(peak_year_idx)
heat = heat.iloc[:, order]

# Transpose so clusters are rows
heat_T = heat.T

# 4) Plot heat map
plt.figure(figsize=(12, 7))
im = plt.imshow(
    heat_T,
    aspect="auto",
    cmap="viridis"
)

plt.colorbar(im, label="Share of AI attention (within year)")

plt.yticks(
    ticks=np.arange(len(heat_T.index)),
    labels=heat_T.index
)
plt.xticks(
    ticks=np.arange(len(heat_T.columns)),
    labels=heat_T.columns,
    rotation=45
)

plt.xlabel("Year")
plt.ylabel("Benefit cluster")
plt.title("AI public benefits over time\n(relative salience within AI discourse)")

plt.tight_layout()
plt.show()

## Stage 7B — Core visualisations (benefits)

Stable core and UMAP embedding space visualisations for the benefit landscape.

In [ ]:
import os
print("CWD:", os.getcwd())
print("ARTIFACT_DIR exists:", os.path.exists("analysis_artifacts"))


In [ ]:
import os

ARTIFACT_DIR = "analysis_artifacts"
expected = "public_dialogue_analysis_v9.xlsx"

matches = []
for root, _, files in os.walk(ARTIFACT_DIR):
    if expected in files:
        matches.append(os.path.join(root, expected))

print("Matches:", matches)


In [ ]:
# @title Visualise the stable core of public technology benefits

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ---- prerequisites ----
if "cluster_entropy" not in globals():
    raise NameError("cluster_entropy not found. Run the cluster entropy section first.")
if "benefits_df" not in globals():
    raise NameError("benefits_df not found. Run benefit extraction first.")

TECH_COL = "technology_meta"
ENTROPY_THRESHOLD = 0.5  # canonical cross-cutting threshold

# ---- build cluster-level dataframe ----
# cluster size = number of benefit phrases
cluster_size = benefits_df["cluster_id"].value_counts()

df = (
    pd.DataFrame({
        "cluster_id": list(cluster_entropy.keys()),
        "entropy": [cluster_entropy[c] for c in cluster_entropy.keys()],
    })
    .set_index("cluster_id")
)

df["size"] = cluster_size
df["size"] = df["size"].fillna(0)
df = df[df["size"] > 0]

# cluster labels (if available)
if "CLUSTER_LABELS" in globals():
    df["label"] = df.index.map(lambda c: CLUSTER_LABELS.get(c, f"Cluster {c}"))
else:
    df["label"] = df.index.map(lambda c: f"Cluster {c}")

entropy_vals = df["entropy"].to_numpy(dtype=float)
size = df["size"].to_numpy(dtype=float)
labels = df["label"].to_numpy(dtype=str)

# ---- define cross-cutting + stable core ----
cross_cutting = entropy_vals >= ENTROPY_THRESHOLD
size_thresh = np.quantile(size, 0.75)  # top 25% by frequency
stable_core = cross_cutting & (size >= size_thresh)

# clusters to annotate (most "core")
score = entropy_vals * np.log1p(size)
annotate_idx = np.argsort(score)[::-1][:12]

# ---- plot ----
plt.figure(figsize=(10, 7))
ax = plt.gca()

# shaded stable-core region
core_region = Rectangle(
    (ENTROPY_THRESHOLD, size_thresh),
    1.0 - ENTROPY_THRESHOLD,
    size.max() * 1.05 - size_thresh,
    alpha=0.12,
    zorder=0
)
ax.add_patch(core_region)

ax.text(
    ENTROPY_THRESHOLD + 0.01,
    size.max() * 1.02,
    "Stable core\n(cross-cutting + high frequency)",
    fontsize=10,
    va="top"
)

plt.scatter(entropy_vals[~cross_cutting], size[~cross_cutting], alpha=0.6, s=35, label="More tech-specific clusters")
plt.scatter(entropy_vals[cross_cutting], size[cross_cutting], alpha=0.8, s=45, label="More cross-cutting clusters")
plt.scatter(entropy_vals[stable_core], size[stable_core], s=90, label="Stable core")

for i in annotate_idx:
    plt.annotate(labels[i], (entropy_vals[i], size[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

plt.axvline(ENTROPY_THRESHOLD, linestyle="--", linewidth=1)
plt.axhline(size_thresh, linestyle="--", linewidth=1)

plt.xlabel("Technology-distribution entropy (higher = more cross-cutting)")
plt.ylabel("Cluster frequency / size (higher = more recurrent)")
plt.title("Figure: The Stable Core of Public Technology Benefits")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# @title Visualise the benefit embedding space
# Clusters projected into 2D using PCA over technology-salience vectors.
# Point size ~ cluster frequency proxy; color ~ framing lens (if available).

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ARTIFACT_DIR = globals().get("ARTIFACT_DIR", "analysis_artifacts")
EXCEL_PATH = globals().get("EXCEL_PATH", os.path.join(ARTIFACT_DIR, "public_dialogue_analysis_v9.xlsx"))

# 1) Load tech x cluster salience (in-memory)
if "salience_df" not in globals():
    raise NameError("salience_df not found. Run Section 6.1a to compute salience_df first.")

tech_by_cluster = salience_df.copy()
tech_by_cluster = tech_by_cluster.apply(pd.to_numeric, errors="coerce").fillna(0)


# Convert to clusters x technologies
cluster_features = tech_by_cluster.T  # rows=clusters, cols=technologies
clusters = cluster_features.index.astype(str).to_numpy()
X = cluster_features.to_numpy(dtype=float)

print("Cluster feature matrix:", X.shape, "(clusters x technologies)")

# 2) Size proxy (use total salience mass across technologies)
size_vals = cluster_features.sum(axis=1).to_numpy()
size_scaled = 30 + 200 * (np.sqrt(size_vals) / (np.sqrt(size_vals).max() + 1e-9))

# 3) Load framing lens mapping if available (optional)
lens_map = None
json_path = os.path.join(ARTIFACT_DIR, "benefit_framing_lens_mappings.json")
if os.path.exists(json_path):
    with open(json_path, "r") as f:
        lens_map = json.load(f)
    # Normalize to {cluster_label: lens}
    if lens_map and all(isinstance(v, list) for v in lens_map.values()):
        inv = {}
        for lens, clist in lens_map.items():
            for c in clist:
                inv[str(c)] = str(lens)
        lens_map = inv
    else:
        lens_map = {str(k): str(v) for k, v in lens_map.items()}

if lens_map is None:
    lens_labels = np.array(["(no lens)"] * len(clusters), dtype=object)
else:
    lens_labels = np.array([lens_map.get(c, "(unmapped)") for c in clusters], dtype=object)

# Encode lenses to integer codes
unique_lenses = pd.unique(lens_labels)
lens_to_code = {lens: i for i, lens in enumerate(unique_lenses)}
codes = np.array([lens_to_code[l] for l in lens_labels], dtype=int)

# 4) PCA projection (standardize across technologies)
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-9
Xz = (X - X_mean) / X_std

from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(Xz)
x, y = coords[:, 0], coords[:, 1]
expl = pca.explained_variance_ratio_

# 5) Plot
plt.figure(figsize=(10, 7))
ax = plt.gca()

# Use a categorical colormap; map integer codes -> colors
cmap = plt.get_cmap("tab20", len(unique_lenses))  # tab20 gives distinct-ish categories
sc = ax.scatter(x, y, s=size_scaled, c=codes, cmap=cmap, alpha=0.85)

ax.set_xlabel(f"Benefit space (PC1; {expl[0]:.0%} var)")
ax.set_ylabel(f"Benefit space (PC2; {expl[1]:.0%} var)")
ax.set_title("Figure: Benefit Space of Public Technology Benefits\n(Clusters projected by technology-salience profile; size ~ frequency proxy)")

# Annotate top-N largest clusters (by size proxy)
topN = 12
top_idx = np.argsort(size_vals)[::-1][:topN]
for i in top_idx:
    ax.annotate(clusters[i], (x[i], y[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

# Legend: show up to 12 most frequent lenses (or all if <= 12)
lens_counts = pd.Series(lens_labels).value_counts()
top_lenses = lens_counts.index[:12].tolist() if len(lens_counts) > 12 else lens_counts.index.tolist()

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markersize=8,
           markerfacecolor=cmap(lens_to_code[lens]), markeredgecolor='none', label=lens)
    for lens in top_lenses
]
ax.legend(handles=legend_handles, title="Framing lens (top shown)", loc="best", frameon=True)

plt.tight_layout()
out_path = "figure_benefit_space_pca.png"
plt.savefig(out_path, dpi=200)
plt.show()

print("Saved figure to:", out_path)

# Part III: Risk–benefit comparison

Having characterised the concern and benefit spaces separately, this part compares them:

- Which technologies have the most skewed concern–benefit balance?
- Are there cross-cutting themes that appear in both concern and benefit clusters
  (indicating genuine ambivalence)?
- How does the AI dialogue balance compare to other technology domains?

In [ ]:
print(concerns_df.columns.tolist())
print(benefits_df.columns.tolist())

In [ ]:
# _volume_table — imported from dialogue_utils


In [ ]:
# _top_clusters — imported from dialogue_utils


If you want per-technology views, filter `concerns_df` / `benefits_df` by `technology` and rerun the top-cluster cell.


# Part IV: Robustness and sensitivity checks

Before drawing conclusions from the clustering results, it is important to test their
stability. This part runs two robustness checks:

1. **k-sensitivity** — how much do the results change if we use 60, 75, or 90 clusters
   instead of the default? Cross-cutting classification and AI-fingerprint rankings should
   be broadly stable if the findings are robust.

2. **Volatility analysis** — which concern/benefit themes have the most unstable rankings
   across k values? Highly volatile clusters may reflect a genuine split that k-means is
   merging differently at different resolutions.


## IV.A Robustness & sensitivity — concerns


## Section 9A — Robustness checks (concerns)

Sensitivity analysis for the concern clustering: alternative k values, AI ranking
volatility, and novelty of AI-specific concerns relative to the broader corpus.

In [ ]:
# clusters_to_labels, clusters_to_lenses, html_escape — imported from dialogue_utils


In [ ]:
# normalized_entropy, ai_fingerprint_over_crosscut — imported from dialogue_utils


In [ ]:
# @title Assess volatility of AI concern rankings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year exists: years × clusters (raw salience)
# years is sorted array

# 1) Relative salience within each year
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# 2) Rank concerns within each year (1 = most salient)
ranks = ai_rel.rank(axis=1, ascending=False, method="average")

# 3) Year-to-year rank change magnitude
rank_diff = ranks.diff().abs()

# Average volatility per year (mean rank change across clusters)
volatility = rank_diff.mean(axis=1)

vol_df = pd.DataFrame({
    "year": volatility.index,
    "mean_rank_change": volatility.values
}).dropna()

# 4) Plot
plt.figure(figsize=(9, 4.5))
plt.plot(vol_df["year"], vol_df["mean_rank_change"], marker="o")

plt.xlabel("Year")
plt.ylabel("Mean absolute rank change")
plt.title("Stability of AI public concerns over time\n(lower values = greater normalisation)")
plt.tight_layout()
plt.show()

display(vol_df)



In [ ]:
# parse_year, tokenize — imported from dialogue_utils


In [ ]:
# @title Assess novelty of concern types in AI dialogues

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year exists: years × clusters (raw salience)
present = (ai_year > 0).astype(int)

seen = set()
rows = []
for y in present.index:
    active = set(present.columns[present.loc[y] > 0])
    new = active - seen
    rows.append({
        "year": int(y),
        "active_clusters": len(active),
        "new_clusters": len(new),
        "new_cluster_share": len(new) / max(len(active), 1)
    })
    seen |= active

nov2 = pd.DataFrame(rows)

plt.figure(figsize=(9, 4.5))
plt.plot(nov2["year"], nov2["new_cluster_share"], marker="o")
plt.xlabel("Year")
plt.ylabel("Share of newly appearing concern clusters")
plt.title("Novelty of concern-types in AI discourse over time")
plt.tight_layout()
plt.show()

display(nov2)


## Section 9A results

The concern sensitivity analysis is complete. Review the outputs above before proceeding to
the benefit sensitivity checks.

In [ ]:
# @title Compare results across alternative cluster counts (60, 75, 90)

show_status("Running sensitivity analysis for cluster counts...")

from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as shannon_entropy

TECH_COL = 'technology_meta'
ks = [60, 75, 90]

# --- Ensure baseline_centroids exist (k=75 baseline) ---
BASELINE_K = 75

if "baseline_centroids" not in globals():
    km_base = KMeans(n_clusters=BASELINE_K, random_state=RANDOM_SEED, n_init="auto")
    _ = km_base.fit_predict(embeddings_normalized)
    baseline_centroids = km_base.cluster_centers_.copy()

# Baseline lens membership: baseline cluster -> set(lenses)
baseline_cluster_to_lenses = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    for cid in data['cluster_ids']:
        baseline_cluster_to_lenses.setdefault(cid, set()).add(lens_name)

# run_for_k replaced by run_sensitivity from dialogue_utils (CIP-0002)

for k in ks:
    print(f"\n=== Sensitivity run: k={k} ===")
    run_sensitivity(
        k, kind='concern',
        embeddings_normalized=embeddings_normalized,
        df=concerns_df,
        output_folder=OUTPUT_FOLDER,
        random_seed=RANDOM_SEED,
        framing_lens_mappings=FRAMING_LENS_MAPPINGS,
    )

show_complete("Sensitivity runs complete (see outputs folder)")


In [ ]:
# parse_year, normalized_entropy, is_privacy_text, entropy_by_year (concerns) — imported from dialogue_utils



## IV.B Robustness & sensitivity — benefits


## Section 9B — Robustness checks (benefits)

The same sensitivity analysis applied to the benefit clustering.

In [ ]:
# clusters_to_labels, clusters_to_lenses, html_escape (benefits) — duplicate removed; imported from dialogue_utils


In [ ]:
# normalized_entropy, ai_fingerprint_over_crosscut (benefits) — duplicate removed; imported from dialogue_utils


In [ ]:
# @title Assess volatility of AI benefit rankings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year exists: years × clusters (raw salience)
# years is sorted array

# 1) Relative salience within each year
ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

# 2) Rank concerns within each year (1 = most salient)
ranks = ai_rel.rank(axis=1, ascending=False, method="average")

# 3) Year-to-year rank change magnitude
rank_diff = ranks.diff().abs()

# Average volatility per year (mean rank change across clusters)
volatility = rank_diff.mean(axis=1)

vol_df = pd.DataFrame({
    "year": volatility.index,
    "mean_rank_change": volatility.values
}).dropna()

# 4) Plot
plt.figure(figsize=(9, 4.5))
plt.plot(vol_df["year"], vol_df["mean_rank_change"], marker="o")

plt.xlabel("Year")
plt.ylabel("Mean absolute rank change")
plt.title("Stability of AI public concerns over time\n(lower values = greater normalisation)")
plt.tight_layout()
plt.show()

display(vol_df)


In [ ]:
# parse_year, tokenize (benefits) — duplicate removed; imported from dialogue_utils


In [ ]:
# @title Assess novelty of concern types in AI dialogues

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year exists: years × clusters (raw salience)
present = (ai_year > 0).astype(int)

seen = set()
rows = []
for y in present.index:
    active = set(present.columns[present.loc[y] > 0])
    new = active - seen
    rows.append({
        "year": int(y),
        "active_clusters": len(active),
        "new_clusters": len(new),
        "new_cluster_share": len(new) / max(len(active), 1)
    })
    seen |= active

nov2 = pd.DataFrame(rows)

plt.figure(figsize=(9, 4.5))
plt.plot(nov2["year"], nov2["new_cluster_share"], marker="o")
plt.xlabel("Year")
plt.ylabel("Share of newly appearing concern clusters")
plt.title("Novelty of concern-types in AI discourse over time")
plt.tight_layout()
plt.show()

display(nov2)


## Section 9B results

The benefit sensitivity analysis is complete. Proceed to Part V to export all outputs.

In [ ]:
# @title Compare results across alternative cluster counts (60, 75, 90)

show_status("Running sensitivity analysis for cluster counts...")

from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as shannon_entropy

TECH_COL = 'technology_meta'
ks = [60, 75, 90]

# --- Ensure baseline_centroids exist (k=75 baseline) ---
BASELINE_K = 75

if "baseline_centroids" not in globals():
    km_base = KMeans(n_clusters=BASELINE_K, random_state=RANDOM_SEED, n_init="auto")
    _ = km_base.fit_predict(embeddings_normalized)
    baseline_centroids = km_base.cluster_centers_.copy()

# Baseline lens membership: baseline cluster -> set(lenses)
baseline_cluster_to_lenses = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    for cid in data['cluster_ids']:
        baseline_cluster_to_lenses.setdefault(cid, set()).add(lens_name)

# run_for_k replaced by run_sensitivity from dialogue_utils (CIP-0002)

for k in ks:
    print(f"\n=== Sensitivity run: k={k} ===")
    run_sensitivity(
        k, kind='benefit',
        embeddings_normalized=benefit_embeddings_normalized,  # CIP-0002 fix: was erroneously concern embeddings
        df=benefits_df,
        output_folder=OUTPUT_FOLDER,
        random_seed=RANDOM_SEED,
        framing_lens_mappings=globals().get('BENEFIT_FRAMING_LENS_MAPPINGS', FRAMING_LENS_MAPPINGS),
    )

show_complete("Sensitivity runs complete (see outputs folder)")


In [ ]:
# parse_year, normalized_entropy, is_privacy_text, entropy_by_year (benefits) — imported from dialogue_utils


# Part V: Export outputs

This cell packages all analysis outputs into a ZIP file for download. The ZIP contains:
- Concern and benefit phrase CSVs with cluster assignments
- Cluster label JSONs
- Extraction diagnostics (yield summary, filter drops, errors)
- Vocabulary frequency tables
- Validation summary and playbook

Run this cell last, after reviewing the robustness checks.

In [ ]:
# @title Export all analysis outputs

import os
import zipfile
from pathlib import Path
from google.colab import files

# Where outputs live
OUTPUT_DIR = Path(OUTPUT_FOLDER)
ARTIFACT_DIR = Path(globals().get("ARTIFACT_DIR", "analysis_artifacts"))

# Generate validation summary (CIP-0005)
generate_validation_summary(
    output_folder=OUTPUT_DIR,
    n_concern_clusters=globals().get("N_CONCERN_CLUSTERS"),
    n_benefit_clusters=globals().get("N_BENEFIT_CLUSTERS"),
)

ZIP_NAME = "public_dialogue_analysis_outputs.zip"
ZIP_PATH = OUTPUT_DIR / ZIP_NAME

print("Preparing ZIP archive...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # Add OUTPUT_FOLDER contents
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file() and path.name != ZIP_NAME:
            zf.write(path, arcname=f"outputs/{path.relative_to(OUTPUT_DIR)}")

    # Optionally add analysis_artifacts if it exists
    if ARTIFACT_DIR.exists():
        for path in ARTIFACT_DIR.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=f"analysis_artifacts/{path.relative_to(ARTIFACT_DIR)}")

    # Bundle the validation playbook (CIP-0005)
    playbook = Path("validation_playbook.md")
    if playbook.exists():
        zf.write(playbook, arcname="validation_playbook.md")
        print("  + validation_playbook.md")

print(f"ZIP written to: {ZIP_PATH}")
print("Files included:")

for p in sorted(OUTPUT_DIR.glob("*")):
    if p.name != ZIP_NAME:
        print(" - outputs/", p.name)

if ARTIFACT_DIR.exists():
    print(" - analysis_artifacts/ (included)")

# Trigger download in Colab
files.download(str(ZIP_PATH))


---
*Analysis complete. Download the ZIP from the file browser (left sidebar in Colab)
or from the `OUTPUT_FOLDER` path shown above.*